# Arena Ranker — Kaggle GPU 训练与推理

本 notebook 在 Kaggle 1提供的 **GPU 环境（T4 16GB / P100 16GB）** 中完成：

1. 安装依赖
2. 设置项目源码
3. 使用 **QLoRA** 微调 `Qwen/Qwen3.5-0.8B` 偏好分类模型
4. 推理并生成 `submission.csv`

### 核心架构

```
prompt + response_a + response_b
        ↓
  apply_chat_template (system + user)
        ↓
  Qwen3.5-0.8B (4-bit NF4 量化)
        ↓
  AutoModelForSequenceClassification
        ↓
  score (分类头, 3-class logits)
```

> **前置准备**
>
> | 项目 | 操作 |
> |------|------|
> | **竞赛数据** | Notebook 右侧 → Add Input → 搜索并添加竞赛数据集 |
> | **GPU** | Settings → Accelerator → 选择 **GPU T4 ×2** 或 **GPU P100** |
> | **联网** | Settings → Internet → **On**（用于下载 HuggingFace 模型）|
>
> 如果不想联网下载模型，请参考最后一节「离线模式」。


## 1. 安装依赖

Kaggle 环境已预装 PyTorch，这里补装 QLoRA 训练所需的包。

In [ ]:
!pip install -q \
    "transformers>=4.55.0" \
    "peft>=0.17.0" \
    "bitsandbytes>=0.45.0" \
    "datasets>=3.0.0" \
    "accelerate>=1.0.0" \
    "scikit-learn>=1.5.0" \
    "tqdm>=4.66.0" \
    "pyyaml>=6.0.2"

## 2. 写入项目源码

将 `arena_ranker` 包的所有源文件写入 `/kaggle/working/arena_ranker/`。

> **替代方案**：也可以把本仓库上传为 Kaggle Dataset，
> 然后 `!pip install /kaggle/input/<your-dataset-slug>/` 来安装。

In [ ]:
import base64
from pathlib import Path

PKG_DIR = Path("/kaggle/working/arena_ranker")
PKG_DIR.mkdir(parents=True, exist_ok=True)

_FILES = {
    "__init__.py": "IiIiQXJlbmEgUmFua2VyIOKAlCBRTG9SQSDlvq7osIMgUXdlbjMuNS0wLjhCIOeUqOS6jiBDaGF0Qm90IEFyZW5hIOWBj+WlvemihOa1i+OAgiIiIgo=",
    "config.py": "IiIiCumFjee9ruaooeWdlyDigJQg5a6a5LmJ6K6t57uD5ZKM5o6o55CG5omA6ZyA55qE5YWo6YOo6buY6K6k5Y+C5pWw44CCCgrmnKzmqKHlnZfph4fnlKggZGF0YWNsYXNzIOe7hOe7h+mFjee9ru+8jOaUr+aMgSBZQU1MIOaMgeS5heWMluOAggrkuLvopoHliIbkuLrkuInpg6jliIbvvJoKICAtIERhdGFDb25maWc6ICDmlbDmja7ot6/lvoTjgIHmlofmnKzmiKrmlq3jgIHpqozor4Hpm4bmr5TkvosKICAtIE1vZGVsQ29uZmlnOiDln7rluqfmqKHlnovjgIHph4/ljJbjgIFMb1JBIOWPguaVsAogIC0gVHJhaW5pbmdDb25maWc6IOWtpuS5oOeOh+OAgWJhdGNoIHNpemXjgIFlcG9jaCDnrYkgVHJhaW5lciDlj4LmlbAKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QsIGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB5YW1sCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5qCH562+55u45YWz5bi46YePCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkxBQkVMX0NPTFVNTlMgPSBbIndpbm5lcl9tb2RlbF9hIiwgIndpbm5lcl9tb2RlbF9iIiwgIndpbm5lcl90aWUiXQpMQUJFTF9UT19JRCA9IHsid2lubmVyX21vZGVsX2EiOiAwLCAid2lubmVyX21vZGVsX2IiOiAxLCAid2lubmVyX3RpZSI6IDJ9CklEX1RPX0xBQkVMID0ge3Y6IGsgZm9yIGssIHYgaW4gTEFCRUxfVE9fSUQuaXRlbXMoKX0KTlVNX0xBQkVMUyA9IDMKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOWvueivneaooeadv+W4uOmHjwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpTWVNURU1fUFJPTVBUID0gIiIiUGxlYXNlIGFjdCBhcyBhbiBpbXBhcnRpYWwganVkZ2UgYW5kIGV2YWx1YXRlIHRoZSBxdWFsaXR5IG9mIHRoZSByZXNwb25zZXMgcHJvdmlkZWQgYnkgdHdvCkFJIGFzc2lzdGFudHMgdG8gdGhlIHVzZXIgcXVlc3Rpb24gZGlzcGxheWVkIGJlbG93LiBZb3Ugc2hvdWxkIGNob29zZSB0aGUgYXNzaXN0YW50IHRoYXQKZm9sbG93cyB0aGUgdXNlcuKAmXMgaW5zdHJ1Y3Rpb25zIGFuZCBhbnN3ZXJzIHRoZSB1c2Vy4oCZcyBxdWVzdGlvbiBiZXR0ZXIuIFlvdXIgZXZhbHVhdGlvbgpzaG91bGQgY29uc2lkZXIgZmFjdG9ycyBzdWNoIGFzIHRoZSBoZWxwZnVsbmVzcywgcmVsZXZhbmNlLCBhY2N1cmFjeSwgZGVwdGgsIGNyZWF0aXZpdHksCmFuZCBsZXZlbCBvZiBkZXRhaWwgb2YgdGhlaXIgcmVzcG9uc2VzLiBCZWdpbiB5b3VyIGV2YWx1YXRpb24gYnkgY29tcGFyaW5nIHRoZSB0d28KcmVzcG9uc2VzIGFuZCBwcm92aWRlIGEgc2hvcnQgZXhwbGFuYXRpb24uIEF2b2lkIGFueSBwb3NpdGlvbiBiaWFzZXMgYW5kIGVuc3VyZSB0aGF0IHRoZQpvcmRlciBpbiB3aGljaCB0aGUgcmVzcG9uc2VzIHdlcmUgcHJlc2VudGVkIGRvZXMgbm90IGluZmx1ZW5jZSB5b3VyIGRlY2lzaW9uLiBEbyBub3QgYWxsb3cKdGhlIGxlbmd0aCBvZiB0aGUgcmVzcG9uc2VzIHRvIGluZmx1ZW5jZSB5b3VyIGV2YWx1YXRpb24uIERvIG5vdCBmYXZvciBjZXJ0YWluIG5hbWVzIG9mCnRoZSBhc3Npc3RhbnRzLiBCZSBhcyBvYmplY3RpdmUgYXMgcG9zc2libGUuIEFmdGVyIHByb3ZpZGluZyB5b3VyIGV4cGxhbmF0aW9uLCBvdXRwdXQgeW91cgpmaW5hbCB2ZXJkaWN0IGJ5IHN0cmljdGx5IGZvbGxvd2luZyB0aGlzIGZvcm1hdDogIltbQV1dIiBpZiBhc3Npc3RhbnQgQSBpcyBiZXR0ZXIsICJbW0JdXSIKaWYgYXNzaXN0YW50IEIgaXMgYmV0dGVyLCBhbmQgIltbQ11dIiBmb3IgYSB0aWUuIiIiCgpVU0VSX1RFTVBMQVRFID0gIntjb252ZXJzYXRpb259IgpWRVJESUNUX1NVRkZJWCA9ICJ2ZXJkaWN0IGlzOiBbWyIKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBEYXRhQ29uZmlnCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgRGF0YUNvbmZpZzoKICAgIHRyYWluX3BhdGg6IHN0ciA9ICJ0cmFpbi5jc3YiCiAgICB0ZXN0X3BhdGg6IHN0ciA9ICJ0ZXN0LmNzdiIKICAgIHRleHRfbWF4X2NoYXJzOiBpbnQgPSA2MDAwCiAgICB2YWxpZGF0aW9uX3NpemU6IGZsb2F0ID0gMC4xCiAgICByYW5kb21fc3RhdGU6IGludCA9IDQyCiAgICBpbmNsdWRlX3N3YXBfdHJhaW46IGJvb2wgPSBUcnVlCiAgICBpbmNsdWRlX3N3YXBfdHRhOiBib29sID0gVHJ1ZQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIE1vZGVsQ29uZmlnCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgTW9kZWxDb25maWc6CiAgICBtb2RlbF9uYW1lOiBzdHIgPSAiUXdlbi9Rd2VuMy41LTAuOEIiCiAgICBjYWNoZV9kaXI6IHN0ciB8IE5vbmUgPSBOb25lCiAgICBsb2NhbF9maWxlc19vbmx5OiBib29sID0gRmFsc2UKICAgIG1heF9sZW5ndGg6IGludCA9IDEwMjQKCiAgICAjIC0tLSA0LWJpdCDph4/ljJYgKFFMb1JBKSAtLS0KICAgIGxvYWRfaW5fNGJpdDogYm9vbCA9IFRydWUKICAgIGJuYl80Yml0X3F1YW50X3R5cGU6IHN0ciA9ICJuZjQiCiAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50OiBib29sID0gVHJ1ZQoKICAgICMgLS0tIExvUkEgLS0tCiAgICB1c2VfbG9yYTogYm9vbCA9IFRydWUKICAgIGxvcmFfcjogaW50ID0gMzIKICAgIGxvcmFfYWxwaGE6IGludCA9IDY0CiAgICBsb3JhX2Ryb3BvdXQ6IGZsb2F0ID0gMC4wNQogICAgbG9yYV9iaWFzOiBzdHIgPSAibm9uZSIKICAgIGxvcmFfdGFyZ2V0X21vZHVsZXM6IGxpc3Rbc3RyXSA9IGZpZWxkKAogICAgICAgIGRlZmF1bHRfZmFjdG9yeT1sYW1iZGE6IFsKICAgICAgICAgICAgInFfcHJvaiIsICJrX3Byb2oiLCAidl9wcm9qIiwgIm9fcHJvaiIsCiAgICAgICAgICAgICJnYXRlX3Byb2oiLCAidXBfcHJvaiIsICJkb3duX3Byb2oiLAogICAgICAgIF0KICAgICkKICAgIGxvcmFfbW9kdWxlc190b19zYXZlOiBsaXN0W3N0cl0gPSBmaWVsZCgKICAgICAgICBkZWZhdWx0X2ZhY3Rvcnk9bGFtYmRhOiBbInNjb3JlIl0KICAgICkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBUcmFpbmluZ0NvbmZpZwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIFRyYWluaW5nQ29uZmlnOgogICAgb3V0cHV0X2Rpcjogc3RyID0gImFydGlmYWN0cy9kZWZhdWx0IgogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAyZS00CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0ID0gMC4wMQogICAgcGVyX2RldmljZV90cmFpbl9iYXRjaF9zaXplOiBpbnQgPSAyCiAgICBwZXJfZGV2aWNlX2V2YWxfYmF0Y2hfc2l6ZTogaW50ID0gNAogICAgZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzOiBpbnQgPSA4CiAgICBudW1fdHJhaW5fZXBvY2hzOiBpbnQgPSAzCiAgICB3YXJtdXBfcmF0aW86IGZsb2F0ID0gMC4xCiAgICB3YXJtdXBfc3RlcHM6IGludCA9IDAKICAgIGxyX3NjaGVkdWxlcl90eXBlOiBzdHIgPSAiY29zaW5lIgogICAgb3B0aW06IHN0ciA9ICJwYWdlZF9hZGFtd18zMmJpdCIKICAgIGZwMTY6IGJvb2wgPSBUcnVlCiAgICBiZjE2OiBib29sID0gRmFsc2UKICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc6IGJvb2wgPSBUcnVlCiAgICBsb2dnaW5nX3N0ZXBzOiBpbnQgPSA1MAogICAgZXZhbF9zdHJhdGVneTogc3RyID0gImVwb2NoIgogICAgc2F2ZV9zdHJhdGVneTogc3RyID0gImVwb2NoIgogICAgc2F2ZV90b3RhbF9saW1pdDogaW50ID0gMgogICAgbG9hZF9iZXN0X21vZGVsX2F0X2VuZDogYm9vbCA9IFRydWUKICAgIG1ldHJpY19mb3JfYmVzdF9tb2RlbDogc3RyID0gImxvZ19sb3NzIgogICAgZ3JlYXRlcl9pc19iZXR0ZXI6IGJvb2wgPSBGYWxzZQogICAgc2VlZDogaW50ID0gNDIKICAgIHJlcG9ydF90bzogc3RyID0gIm5vbmUiCiAgICBkYXRhbG9hZGVyX251bV93b3JrZXJzOiBpbnQgPSAwCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAgQXBwQ29uZmlnICjpobblsYLogZrlkIgpCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgQXBwQ29uZmlnOgogICAgZGF0YTogRGF0YUNvbmZpZyA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1EYXRhQ29uZmlnKQogICAgbW9kZWw6IE1vZGVsQ29uZmlnID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PU1vZGVsQ29uZmlnKQogICAgdHJhaW5pbmc6IFRyYWluaW5nQ29uZmlnID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PVRyYWluaW5nQ29uZmlnKQoKICAgIGRlZiB0b19kaWN0KHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBOb25lOgogICAgICAgIG91dHB1dF9wYXRoID0gUGF0aChwYXRoKQogICAgICAgIG91dHB1dF9wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgb3V0cHV0X3BhdGgud3JpdGVfdGV4dCgKICAgICAgICAgICAgeWFtbC5zYWZlX2R1bXAoc2VsZi50b19kaWN0KCksIHNvcnRfa2V5cz1GYWxzZSksCiAgICAgICAgICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQoKCmRlZiBsb2FkX2NvbmZpZyhwYXRoOiBzdHIgfCBQYXRoIHwgTm9uZSA9IE5vbmUpIC0+IEFwcENvbmZpZzoKICAgICIiIuS7jiBZQU1MIOaWh+S7tuWKoOi9vemFjee9ru+8m3BhdGg9Tm9uZSDml7bov5Tlm57lhajpg6jpu5jorqTlgLzjgIIiIiIKICAgIGlmIHBhdGggaXMgTm9uZToKICAgICAgICByZXR1cm4gQXBwQ29uZmlnKCkKICAgIHJhdyA9IHlhbWwuc2FmZV9sb2FkKFBhdGgocGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgdHJhaW5pbmdfcmF3ID0gcmF3LmdldCgidHJhaW5pbmciLCB7fSkKCiAgICAjIOWFvOWuueaXp+mFjee9rjog5pu+6K+v5oqKIDAuMSDov5nnsbvmr5Tkvovlhpnov5sgd2FybXVwX3N0ZXBz44CCCiAgICBsZWdhY3lfd2FybXVwID0gdHJhaW5pbmdfcmF3LmdldCgid2FybXVwX3N0ZXBzIikKICAgIGlmICJ3YXJtdXBfcmF0aW8iIG5vdCBpbiB0cmFpbmluZ19yYXcgYW5kIGlzaW5zdGFuY2UobGVnYWN5X3dhcm11cCwgZmxvYXQpOgogICAgICAgIGlmIDAuMCA8PSBsZWdhY3lfd2FybXVwIDw9IDEuMDoKICAgICAgICAgICAgdHJhaW5pbmdfcmF3WyJ3YXJtdXBfcmF0aW8iXSA9IGxlZ2FjeV93YXJtdXAKICAgICAgICAgICAgdHJhaW5pbmdfcmF3WyJ3YXJtdXBfc3RlcHMiXSA9IDAKCiAgICByZXR1cm4gQXBwQ29uZmlnKAogICAgICAgIGRhdGE9RGF0YUNvbmZpZygqKnJhdy5nZXQoImRhdGEiLCB7fSkpLAogICAgICAgIG1vZGVsPU1vZGVsQ29uZmlnKCoqcmF3LmdldCgibW9kZWwiLCB7fSkpLAogICAgICAgIHRyYWluaW5nPVRyYWluaW5nQ29uZmlnKCoqdHJhaW5pbmdfcmF3KSwKICAgICkK",
    "data.py": "IiIiCuaVsOaNruWkhOeQhuaooeWdlyDigJQg5Yqg6L29IENTVuOAgeaWh+acrOa4hea0l+OAgWNoYXQgdGVtcGxhdGUgdG9rZW5pemF0aW9u44CBbWV0cmljcyDorqHnrpfjgIIKCuaguOW/g+a1geeoi++8mgogIDEuIOivu+WPliBDU1Yg4oaSIOa4hea0l+aWh+acrOWtl+autSAo6Kej5p6QIEpTT04g5pWw57uEIC8gUHl0aG9uIGxpc3Qg562J5qC85byPKQogIDIuIOS9v+eUqCB0b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgpIOaehOW7uiBzeXN0ZW0gKyB1c2VyIOWvueivnQogIDMuIOmAmui/hyBEYXRhc2V0Lm1hcCgpIOWujOaIkCB0b2tlbml6YXRpb27vvIzovpPlh7ogaW5wdXRfaWRzIC8gYXR0ZW50aW9uX21hc2sgLyBsYWJlbHMKICA0LiDmj5DkvpsgY29tcHV0ZV9tZXRyaWNzKCkg5L6bIFRyYWluZXIg5L2/55SoCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdAppbXBvcnQganNvbgpmcm9tIGl0ZXJ0b29scyBpbXBvcnQgemlwX2xvbmdlc3QKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBkYXRhc2V0cyBpbXBvcnQgRGF0YXNldApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmUsIGxvZ19sb3NzCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKCmZyb20gYXJlbmFfcmFua2VyLmNvbmZpZyBpbXBvcnQgKAogICAgTEFCRUxfQ09MVU1OUywKICAgIExBQkVMX1RPX0lELAogICAgU1lTVEVNX1BST01QVCwKICAgIFVTRVJfVEVNUExBVEUsCiAgICBWRVJESUNUX1NVRkZJWCwKICAgIERhdGFDb25maWcsCikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDmlofmnKzop6PmnpAgJiDmuIXmtJcKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlOiBBbnkpIC0+IGxpc3Rbc3RyXToKICAgICIiIgogICAg5bCG5Y6f5aeLIENTViDlrZfmrrXop6PmnpDkuLrlrZfnrKbkuLLliJfooajjgIIKICAgIOaUr+aMgeS4ieenjei+k+WFpeagvOW8j++8mgogICAgICAtIOaZrumAmuWtl+espuS4siDihpIg55u05o6l5YyF5YWl5YiX6KGoCiAgICAgIC0gSlNPTiDmlbDnu4TlrZfnrKbkuLIg4oaSIGpzb24ubG9hZHMg6Kej5p6QCiAgICAgIC0gUHl0aG9uIGxpc3Qg5a2X56ym5LiyIOKGkiBhc3QubGl0ZXJhbF9ldmFsIOino+aekAogICAgIiIiCiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KToKICAgICAgICByZXR1cm4gW3N0cihpdGVtKSBmb3IgaXRlbSBpbiB2YWx1ZV0KICAgIGlmIHZhbHVlIGlzIE5vbmUgb3IgKGlzaW5zdGFuY2UodmFsdWUsIGZsb2F0KSBhbmQgcGQuaXNuYSh2YWx1ZSkpOgogICAgICAgIHJldHVybiBbXQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIHN0cik6CiAgICAgICAgcmV0dXJuIFtzdHIodmFsdWUpXQogICAgdGV4dCA9IHZhbHVlLnN0cmlwKCkKICAgIGlmIG5vdCB0ZXh0OgogICAgICAgIHJldHVybiBbXQogICAgZm9yIHBhcnNlciBpbiAoanNvbi5sb2FkcywgYXN0LmxpdGVyYWxfZXZhbCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBwYXJzZXIodGV4dCkKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwYXJzZWQsIGxpc3QpOgogICAgICAgICAgICAgICAgcmV0dXJuIFtzdHIoaXRlbSkgZm9yIGl0ZW0gaW4gcGFyc2VkXQogICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgU3ludGF4RXJyb3IpOgogICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIFt0ZXh0XQoKCmRlZiBfbm9ybWFsaXplX2NvbnZlcnNhdGlvbl90dXJucyh2YWx1ZTogQW55LCBtYXhfY2hhcnM6IGludCkgLT4gbGlzdFtzdHJdOgogICAgIiIi5bCG5Y6f5aeL5a2X5q616L2s5Li65oyJ6L2u5qyh5L+d5bqP55qE5a2X56ym5Liy5YiX6KGo77yM5bm25Zyo5oC75a2X56ym6aKE566X5YaF5oiq5pat44CCIiIiCiAgICBjaHVua3MgPSBfcGFyc2VfY29udmVyc2F0aW9uX2ZpZWxkKHZhbHVlKQogICAgbm9ybWFsaXplZCA9IFtjaHVuay5zdHJpcCgpIGZvciBjaHVuayBpbiBjaHVua3MgaWYgc3RyKGNodW5rKS5zdHJpcCgpXQogICAgaWYgbm90IG5vcm1hbGl6ZWQ6CiAgICAgICAgcmV0dXJuIFtdCgogICAga2VwdDogbGlzdFtzdHJdID0gW10KICAgIHVzZWRfY2hhcnMgPSAwCiAgICBmb3IgY2h1bmsgaW4gbm9ybWFsaXplZDoKICAgICAgICByZW1haW5pbmcgPSBtYXhfY2hhcnMgLSB1c2VkX2NoYXJzCiAgICAgICAgaWYgcmVtYWluaW5nIDw9IDA6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgdHJ1bmNhdGVkID0gY2h1bmtbOnJlbWFpbmluZ10KICAgICAgICBpZiB0cnVuY2F0ZWQ6CiAgICAgICAgICAgIGtlcHQuYXBwZW5kKHRydW5jYXRlZCkKICAgICAgICAgICAgdXNlZF9jaGFycyArPSBsZW4odHJ1bmNhdGVkKQogICAgcmV0dXJuIGtlcHQKCgpkZWYgX2J1aWxkX2NvbnZlcnNhdGlvbl90ZXh0KAogICAgcHJvbXB0X3R1cm5zOiBsaXN0W3N0cl0sCiAgICByZXNwb25zZV9hX3R1cm5zOiBsaXN0W3N0cl0sCiAgICByZXNwb25zZV9iX3R1cm5zOiBsaXN0W3N0cl0sCikgLT4gc3RyOgogICAgIiIi5oyJ6L2u5qyh5Lqk6ZSZ5ou85o6lIHByb21wdCAvIEEgLyBC77yM5L+d55WZ5aSa6L2u5a+56K+d57uT5p6E44CCIiIiCiAgICBoZWFkID0gIjx8VGhlIFN0YXJ0IG9mIENvbnZlcnNhdGlvbiBiZXR3ZWVuIGEgVXNlciBhbmQgdHdvIEFzc2lzdGFudHN8PiIKICAgIHRhaWwgPSAiPHxUaGUgRW5kIG9mIENvbnZlcnNhdGlvbiBiZXR3ZWVuIGEgVXNlciBhbmQgdHdvIEFzc2lzdGFudHN8PlxuIgogICAgcGFydHMgPSBbXQogICAgZm9yIHByb21wdCwgcmVzcG9uc2VfYSwgcmVzcG9uc2VfYiBpbiB6aXBfbG9uZ2VzdCgKICAgICAgICBwcm9tcHRfdHVybnMsCiAgICAgICAgcmVzcG9uc2VfYV90dXJucywKICAgICAgICByZXNwb25zZV9iX3R1cm5zLAogICAgICAgIGZpbGx2YWx1ZT0ibnVsbCIsCiAgICApOgogICAgICAgIHBhcnRzLmFwcGVuZCgKICAgICAgICAgICAgZiJcbiMjIyBVc2VyOlxue3Byb21wdH1cblxuIyMjIEFzc2lzdGFudCBBOlxue3Jlc3BvbnNlX2F9XG5cbiMjIyBBc3Npc3RhbnQgQjpcbntyZXNwb25zZV9ifVxuIgogICAgICAgICkKICAgIHJldHVybiBoZWFkICsgIiIuam9pbihwYXJ0cykgKyB0YWlsCgoKZGVmIF9idWlsZF9sYWJlbChyb3c6IHBkLlNlcmllcykgLT4gaW50OgogICAgIiIi5LuOIG9uZS1ob3Qg5qCH562+5YiXIOKGkiDljZXkuIDmlbTmlbDmoIfnrb4gKDA9QeiDnCwgMT1C6IOcLCAyPeW5s+WxgCnjgIIiIiIKICAgIGZvciBjb2wsIGxhYmVsX2lkIGluIExBQkVMX1RPX0lELml0ZW1zKCk6CiAgICAgICAgaWYgaW50KHJvd1tjb2xdKSA9PSAxOgogICAgICAgICAgICByZXR1cm4gbGFiZWxfaWQKICAgIHJhaXNlIFZhbHVlRXJyb3IoZiLml6DmlYjmoIfnrb7ooYw6IHtyb3dbTEFCRUxfQ09MVU1OU10udG9fZGljdCgpfSIpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5pWw5o2u5Yqg6L29ICYg6aKE5aSE55CGCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbG9hZF9hbmRfcHJlcHJvY2VzcygKICAgIGNzdl9wYXRoOiBzdHIsCiAgICBtYXhfY2hhcnM6IGludCA9IDYwMDAsCiAgICBpc190cmFpbjogYm9vbCA9IFRydWUsCikgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiCiAgICDliqDovb0gQ1NWIOW5tumihOWkhOeQhuaWh+acrOWtl+auteOAggoKICAgIFJldHVybnM6CiAgICAgICAgRGF0YUZyYW1l77yM5YyF5ZCrIGlkIC8gcHJvbXB0X3R1cm5zIC8gcmVzcG9uc2VfYV90dXJucyAvIHJlc3BvbnNlX2JfdHVybnMKICAgICAgICDku6Xlj4ogKOS7heiuree7g+mbhikgbGFiZWxzIOWIl+OAggogICAgIiIiCiAgICBkZiA9IHBkLnJlYWRfY3N2KGNzdl9wYXRoKQogICAgZGZbInByb21wdF90dXJucyJdID0gZGZbInByb21wdCJdLm1hcCgKICAgICAgICBsYW1iZGEgeDogX25vcm1hbGl6ZV9jb252ZXJzYXRpb25fdHVybnMoeCwgbWF4X2NoYXJzKQogICAgKQogICAgZGZbInJlc3BvbnNlX2FfdHVybnMiXSA9IGRmWyJyZXNwb25zZV9hIl0ubWFwKAogICAgICAgIGxhbWJkYSB4OiBfbm9ybWFsaXplX2NvbnZlcnNhdGlvbl90dXJucyh4LCBtYXhfY2hhcnMpCiAgICApCiAgICBkZlsicmVzcG9uc2VfYl90dXJucyJdID0gZGZbInJlc3BvbnNlX2IiXS5tYXAoCiAgICAgICAgbGFtYmRhIHg6IF9ub3JtYWxpemVfY29udmVyc2F0aW9uX3R1cm5zKHgsIG1heF9jaGFycykKICAgICkKICAgIGlmIGlzX3RyYWluOgogICAgICAgIGRmWyJsYWJlbHMiXSA9IGRmLmFwcGx5KF9idWlsZF9sYWJlbCwgYXhpcz0xKQogICAgcmV0dXJuIGRmCgoKZGVmIHNwbGl0X3RyYWluX3ZhbGlkKAogICAgZGY6IHBkLkRhdGFGcmFtZSwKICAgIGNvbmZpZzogRGF0YUNvbmZpZywKKSAtPiB0dXBsZVtwZC5EYXRhRnJhbWUsIHBkLkRhdGFGcmFtZV06CiAgICAiIiLmjIkgc3RyYXRpZmllZCBzcGxpdCDliJLliIborq3nu4Ppm4blkozpqozor4Hpm4bjgIIiIiIKICAgIHRyYWluX2RmLCB2YWxpZF9kZiA9IHRyYWluX3Rlc3Rfc3BsaXQoCiAgICAgICAgZGYsCiAgICAgICAgdGVzdF9zaXplPWNvbmZpZy52YWxpZGF0aW9uX3NpemUsCiAgICAgICAgcmFuZG9tX3N0YXRlPWNvbmZpZy5yYW5kb21fc3RhdGUsCiAgICAgICAgc3RyYXRpZnk9ZGZbImxhYmVscyJdLAogICAgKQogICAgcmV0dXJuIHRyYWluX2RmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSksIHZhbGlkX2RmLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDaGF0IFRlbXBsYXRlIFRva2VuaXphdGlvbgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIF9idWlsZF9jaGF0X21lc3NhZ2VzKGNvbnZlcnNhdGlvbjogc3RyKSAtPiBsaXN0W2RpY3RdOgogICAgIiIiCiAgICDmnoTlu7rlr7nor53mtojmga/liJfooajvvIznlKjkuo4gYXBwbHlfY2hhdF90ZW1wbGF0ZeOAggogICAgICAtIHN5c3RlbTog6K+E5aeU6KeS6Imy5oyH5LukCiAgICAgIC0gdXNlcjogICDkv53nlZnova7mrKHnu5PmnoTnmoTlrozmlbTlr7nor50KICAgICIiIgogICAgcmV0dXJuIFsKICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1fUFJPTVBUfSwKICAgICAgICB7CiAgICAgICAgICAgICJyb2xlIjogInVzZXIiLAogICAgICAgICAgICAiY29udGVudCI6IFVTRVJfVEVNUExBVEUuZm9ybWF0KGNvbnZlcnNhdGlvbj1jb252ZXJzYXRpb24pLAogICAgICAgIH0sCiAgICBdCgoKZGVmIF90b2tlbml6ZV9zaW5nbGUoZXhhbXBsZTogZGljdCwgdG9rZW5pemVyLCBtYXhfbGVuZ3RoOiBpbnQpIC0+IGRpY3Q6CiAgICAiIiIKICAgIOWvueWNleadoeagt+acrOWBmiB0b2tlbml6YXRpb27vvIjkvpsgRGF0YXNldC5tYXAg6LCD55So77yJ44CCCgogICAg5q2l6aqk77yaCiAgICAgIDEuIOeUqCBhcHBseV9jaGF0X3RlbXBsYXRlIOWwhuWvueivneagvOW8j+WMluS4uuaWh+acrAogICAgICAyLiDnlKggdG9rZW5pemVyIOe8lueggeW5tuaIquaWreWIsCBtYXhfbGVuZ3RoCiAgICAiIiIKICAgIGNvbnZlcnNhdGlvbiA9IF9idWlsZF9jb252ZXJzYXRpb25fdGV4dCgKICAgICAgICBleGFtcGxlWyJwcm9tcHRfdHVybnMiXSwKICAgICAgICBleGFtcGxlWyJyZXNwb25zZV9hX3R1cm5zIl0sCiAgICAgICAgZXhhbXBsZVsicmVzcG9uc2VfYl90dXJucyJdLAogICAgKQogICAgbWVzc2FnZXMgPSBfYnVpbGRfY2hhdF9tZXNzYWdlcyhjb252ZXJzYXRpb24pCiAgICAjIOWFiOW+l+WIsOagvOW8j+WMluaWh+acrO+8jOWGjeWNleeLrCB0b2tlbml6ZSDku6Xnsr7noa7mjqfliLbmiKrmlq0KICAgIHRleHQgPSB0b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICBtZXNzYWdlcywKICAgICAgICB0b2tlbml6ZT1GYWxzZSwKICAgICAgICBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwKICAgICkgKyBWRVJESUNUX1NVRkZJWAogICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICB0ZXh0LAogICAgICAgIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICBtYXhfbGVuZ3RoPW1heF9sZW5ndGgsCiAgICAgICAgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlLCAgIyBjaGF0IHRlbXBsYXRlIOW3sue7j+WMheWQq+aJgOacieeJueauiiB0b2tlbgogICAgKQogICAgcmV0dXJuIGVuY29kZWQKCgpkZWYgYnVpbGRfZGF0YXNldCgKICAgIGRmOiBwZC5EYXRhRnJhbWUsCiAgICB0b2tlbml6ZXIsCiAgICBtYXhfbGVuZ3RoOiBpbnQgPSAxMDI0LAogICAgaXNfdHJhaW46IGJvb2wgPSBUcnVlLAogICAgaW5jbHVkZV9zd2FwOiBib29sID0gRmFsc2UsCiAgICBzd2FwX3BhaXJzOiBib29sID0gRmFsc2UsCikgLT4gRGF0YXNldDoKICAgICIiIgogICAg5LuOIERhdGFGcmFtZSDmnoTlu7ogdG9rZW5pemVkIEh1Z2dpbmdGYWNlIERhdGFzZXTjgIIKCiAgICDovpPlh7rliJfvvIjorq3nu4PvvIk6IGlucHV0X2lkcywgYXR0ZW50aW9uX21hc2ssIGxhYmVscwogICAg6L6T5Ye65YiX77yI5rWL6K+V77yJOiBpbnB1dF9pZHMsIGF0dGVudGlvbl9tYXNrCiAgICAiIiIKICAgIGNvbHMgPSBbImlkIiwgInByb21wdF90dXJucyIsICJyZXNwb25zZV9hX3R1cm5zIiwgInJlc3BvbnNlX2JfdHVybnMiXQogICAgaWYgaXNfdHJhaW46CiAgICAgICAgY29scy5hcHBlbmQoImxhYmVscyIpCiAgICBiYXNlX2RmID0gZGZbY29sc10uY29weSgpCgogICAgaWYgc3dhcF9wYWlycyBvciBpbmNsdWRlX3N3YXA6CiAgICAgICAgc3dhcF9kZiA9IGJhc2VfZGYuY29weSgpCiAgICAgICAgc3dhcF9kZlsicmVzcG9uc2VfYV90dXJucyJdID0gYmFzZV9kZlsicmVzcG9uc2VfYl90dXJucyJdCiAgICAgICAgc3dhcF9kZlsicmVzcG9uc2VfYl90dXJucyJdID0gYmFzZV9kZlsicmVzcG9uc2VfYV90dXJucyJdCiAgICAgICAgaWYgaXNfdHJhaW46CiAgICAgICAgICAgIHN3YXBfZGZbImxhYmVscyJdID0gc3dhcF9kZlsibGFiZWxzIl0ubWFwKAogICAgICAgICAgICAgICAgbGFtYmRhIHg6IDEgaWYgeCA9PSAwIGVsc2UgMCBpZiB4ID09IDEgZWxzZSB4CiAgICAgICAgICAgICkKICAgICAgICBpZiBzd2FwX3BhaXJzOgogICAgICAgICAgICBiYXNlX2RmID0gc3dhcF9kZgogICAgICAgIGVsaWYgaW5jbHVkZV9zd2FwOgogICAgICAgICAgICBiYXNlX2RmID0gcGQuY29uY2F0KFtiYXNlX2RmLCBzd2FwX2RmXSwgaWdub3JlX2luZGV4PVRydWUpCgogICAgZGF0YXNldCA9IERhdGFzZXQuZnJvbV9wYW5kYXMoYmFzZV9kZiwgcHJlc2VydmVfaW5kZXg9RmFsc2UpCgogICAgZGF0YXNldCA9IGRhdGFzZXQubWFwKAogICAgICAgIGxhbWJkYSB4OiBfdG9rZW5pemVfc2luZ2xlKHgsIHRva2VuaXplciwgbWF4X2xlbmd0aCksCiAgICAgICAgZGVzYz0iVG9rZW5pemluZyIsCiAgICApCgogICAgIyDnp7vpmaTmlofmnKzliJfvvIzlj6rkv53nlZnmqKHlnovpnIDopoHnmoTmlbDlgLzliJcKICAgIHJlbW92ZV9jb2xzID0gWyJwcm9tcHRfdHVybnMiLCAicmVzcG9uc2VfYV90dXJucyIsICJyZXNwb25zZV9iX3R1cm5zIiwgImlkIl0KICAgIGRhdGFzZXQgPSBkYXRhc2V0LnJlbW92ZV9jb2x1bW5zKAogICAgICAgIFtjIGZvciBjIGluIHJlbW92ZV9jb2xzIGlmIGMgaW4gZGF0YXNldC5jb2x1bW5fbmFtZXNdCiAgICApCiAgICByZXR1cm4gZGF0YXNldAoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIE1ldHJpY3MKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBfc29mdG1heCh4OiBucC5uZGFycmF5LCBheGlzOiBpbnQgPSAtMSkgLT4gbnAubmRhcnJheToKICAgICIiIuaVsOWAvOeos+WumueahCBzb2Z0bWF4IOWunueOsO+8iOmBv+WFjeW8leWFpSBzY2lweSDkvp3otZbvvInjgIIiIiIKICAgIGVfeCA9IG5wLmV4cCh4IC0gbnAubWF4KHgsIGF4aXM9YXhpcywga2VlcGRpbXM9VHJ1ZSkpCiAgICByZXR1cm4gZV94IC8gZV94LnN1bShheGlzPWF4aXMsIGtlZXBkaW1zPVRydWUpCgoKZGVmIGNvbXB1dGVfbWV0cmljcyhldmFsX3ByZWQpIC0+IGRpY3Rbc3RyLCBmbG9hdF06CiAgICAiIiIKICAgIFRyYWluZXIg55qEIGNvbXB1dGVfbWV0cmljcyDlm57osIPjgIIKCiAgICDorqHnrpc6CiAgICAgIC0gbG9nX2xvc3M6IOamgueOh+e7j+ijgeWJquWQjuiuoeeul++8jOmBv+WFjSBsb2coMCkg5a+86Ie05p6B56uv5YC8CiAgICAgIC0gYWNjdXJhY3k6IOWIhuexu+WHhuehrueOhwogICAgIiIiCiAgICBsb2dpdHMsIGxhYmVscyA9IGV2YWxfcHJlZAoKICAgICMg5aSE55CG5Y+v6IO955qEIE5hTu+8iENQVSDmqKHlvI/miJbmlbDlgLzkuI3nqLPlrprml7blj6/og73lh7rnjrDvvIkKICAgIG5hbl9tYXNrID0gbnAuaXNuYW4obG9naXRzKS5hbnkoYXhpcz0tMSkKICAgIGlmIG5hbl9tYXNrLmFueSgpOgogICAgICAgIGxvZ2l0cyA9IG5wLmNvcHkobG9naXRzKQogICAgICAgIGxvZ2l0c1tuYW5fbWFza10gPSAwLjAgICMgTmFOIOihjOabv+aNouS4uuWdh+WMgOWIhuW4gwoKICAgICMgbG9naXRzIOKGkiDmpoLnjocKICAgIHByb2JzID0gX3NvZnRtYXgobG9naXRzLCBheGlzPS0xKQoKICAgICMg5qaC546H6KOB5YmqICjlr7kgbG9nX2xvc3Mg6K+E5YiG6Z2e5bi46YeN6KaBKQogICAgZXBzID0gMWUtNwogICAgcHJvYnMgPSBucC5jbGlwKHByb2JzLCBlcHMsIDEuMCAtIGVwcykKICAgIHByb2JzID0gcHJvYnMgLyBwcm9icy5zdW0oYXhpcz0xLCBrZWVwZGltcz1UcnVlKQoKICAgIGxvZ2xvc3MgPSBmbG9hdChsb2dfbG9zcyhsYWJlbHMsIHByb2JzLCBsYWJlbHM9WzAsIDEsIDJdKSkKICAgIHByZWRzID0gbnAuYXJnbWF4KGxvZ2l0cywgYXhpcz0tMSkKICAgIGFjYyA9IGZsb2F0KGFjY3VyYWN5X3Njb3JlKGxhYmVscywgcHJlZHMpKQoKICAgIHJldHVybiB7ImxvZ19sb3NzIjogbG9nbG9zcywgImFjY3VyYWN5IjogYWNjfQo=",
    "hf.py": "IiIiCuaooeWei+WKoOi9veaooeWdlyDigJQg6LSf6LSjIHRva2VuaXplciDlkowgUUxvUkEg5YiG57G75qih5Z6L55qE5Yqg6L2944CCCgrmoLjlv4PnrZbnlaXvvJoKICAxLiBUb2tlbml6ZXI6IOWKoOi9veWQjuajgOafpSBwYWRfdG9rZW7vvIzoi6XnvLrlpLHliJnorr7kuLogZW9zX3Rva2VuCiAgMi4g5qih5Z6LOiDpgJrov4cgQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAzLiBMb1JBOiDkvb/nlKggUEVGVCDnmoQgTG9yYUNvbmZpZyDms6jlhaUgYWRhcHRlcu+8jOWQjOaXtuS/neWtmOWIhuexu+WktCAoc2NvcmUpCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKZnJvbSBwZWZ0IGltcG9ydCBMb3JhQ29uZmlnLCBUYXNrVHlwZSwgZ2V0X3BlZnRfbW9kZWwsIHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0ICgKICAgIEF1dG9Db25maWcsCiAgICBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLAogICAgQXV0b1Rva2VuaXplciwKICAgIEJpdHNBbmRCeXRlc0NvbmZpZywKKQoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBNb2RlbENvbmZpZywgTlVNX0xBQkVMUwoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyZW5hX3Jhbmtlci5oZiIpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg6L6F5Yqp5Ye95pWwCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2Rlc2NyaWJlX21vZGVsX3NvdXJjZShtb2RlbF9uYW1lOiBzdHIpIC0+IHN0cjoKICAgICIiIuaPj+i/sOaooeWei+adpea6kO+8jOaWueS+v+aOkuafpeWKoOi9vemXrumimOOAgiIiIgogICAgc291cmNlX3BhdGggPSBQYXRoKG1vZGVsX25hbWUpLmV4cGFuZHVzZXIoKQogICAgaWYgc291cmNlX3BhdGguZXhpc3RzKCk6CiAgICAgICAgY29uZmlnX3BhdGggPSBzb3VyY2VfcGF0aCAvICJjb25maWcuanNvbiIKICAgICAgICBpZiBjb25maWdfcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgcmV0dXJuIGYi5qOA5rWL5Yiw5pys5Zyw5qih5Z6L55uu5b2V77yae3NvdXJjZV9wYXRofSIKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIuajgOa1i+WIsOWQjOWQjeacrOWcsOebruW9lSB7c291cmNlX3BhdGh977yM5L2G57y65bCRIGNvbmZpZy5qc29u44CCIgogICAgICAgICAgICAidHJhbnNmb3JtZXJzIOS8muaKiuWug+W9k+aIkOacrOWcsOaooeWei+i3r+W+hOW5tuebtOaOpeWKoOi9veWksei0peOAgiIKICAgICAgICApCiAgICBpZiAiLyIgaW4gbW9kZWxfbmFtZToKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmImB7bW9kZWxfbmFtZX1gIOWwhuiiq+W9k+S9nCBIdWdnaW5nRmFjZSDku5PlupMgSUTjgIIiCiAgICAgICAgICAgICLpppbmrKHov5DooYzpnIDogZTnvZHkuIvovb3vvJvnprvnur/njq/looPor7fpooTkuIvovb3liLDmnKzlnLDlho3kv67mlLkgbW9kZWxfbmFtZeOAgiIKICAgICAgICApCiAgICByZXR1cm4gZiJge21vZGVsX25hbWV9YCDml6LkuI3mmK/mnKzlnLDnm67lvZXvvIzkuZ/kuI3mmK/moIflh4YgSHVnZ2luZ0ZhY2Ug5LuT5bqTIElE44CCIgoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIFRva2VuaXplcgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIGxvYWRfdG9rZW5pemVyKGNvbmZpZzogTW9kZWxDb25maWcpOgogICAgIiIiCiAgICDliqDovb0gdG9rZW5pemVyIOW5tumFjee9riBwYWRfdG9rZW7jgIIKCiAgICBRd2VuIOezu+WIl+aooeWei+mAmuW4uOayoeaciem7mOiupCBwYWRfdG9rZW7vvIzov5nph4zlsIblhbborr7kuLogZW9zX3Rva2Vu44CCCiAgICDlkIzml7borr7nva4gcGFkZGluZ19zaWRlPSJsZWZ0Iu+8iGRlY29kZXItb25seSDmqKHlnovlgZrliIbnsbvml7bnmoTmjqjojZDlgZrms5XvvInjgIIKICAgICIiIgogICAgdHJ5OgogICAgICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBjb25maWcubW9kZWxfbmFtZSwKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9Y29uZmlnLmxvY2FsX2ZpbGVzX29ubHksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb0gdG9rZW5pemVyIOWksei0pToge2V4Y31cbiIKICAgICAgICAgICAgZiJ7X2Rlc2NyaWJlX21vZGVsX3NvdXJjZShjb25maWcubW9kZWxfbmFtZSl9IgogICAgICAgICkgZnJvbSBleGMKCiAgICAjIOiuvue9riBwYWRfdG9rZW7vvIhRd2VuIOmAmuW4uOayoeaciem7mOiupOWAvO+8iQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBMT0dHRVIuaW5mbygKICAgICAgICAgICAgIlRva2VuaXplciDnvLrlsJEgcGFkX3Rva2Vu77yM5bey6K6+5Li6IGVvc190b2tlbjogJyVzJyAoaWQ9JXMpIiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiwKICAgICAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCwKICAgICAgICApCgogICAgIyBkZWNvZGVyLW9ubHkg5qih5Z6L5YGa5YiG57G75pe277yM5bem5L6n5aGr5YWF5Y+v6YG/5YWN5pyA5ZCO5LiA5LiqIHRva2VuIOS4uiBwYWQKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCiAgICByZXR1cm4gdG9rZW5pemVyCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg6YeP5YyW6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2JuYl9jb25maWcoY29uZmlnOiBNb2RlbENvbmZpZykgLT4gQml0c0FuZEJ5dGVzQ29uZmlnIHwgTm9uZToKICAgICIiIuaehOW7uiBCaXRzQW5kQnl0ZXNDb25maWcg55So5LqOIDQtYml0IFFMb1JBIOmHj+WMluOAgiIiIgogICAgaWYgbm90IGNvbmZpZy5sb2FkX2luXzRiaXQ6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgICMgNC1iaXQg6YeP5YyW6ZyA6KaBIENVREHvvJtDUFUg5qih5byP5LiL6Lez6L+HCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBMT0dHRVIud2FybmluZygi5pyq5qOA5rWL5YiwIENVREHvvIzot7Pov4cgNC1iaXQg6YeP5YyW77yI5bCG5LulIGZsb2F0MzIg5Yqg6L295qih5Z6L77yJIikKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPWNvbmZpZy5ibmJfNGJpdF9xdWFudF90eXBlLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50PWNvbmZpZy5ibmJfNGJpdF91c2VfZG91YmxlX3F1YW50LAogICAgKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIExvUkEg6YWN572uCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgX2J1aWxkX2xvcmFfY29uZmlnKGNvbmZpZzogTW9kZWxDb25maWcpIC0+IExvcmFDb25maWc6CiAgICAiIiIKICAgIOaehOW7uiBMb1JBIOmAgumFjeWZqOmFjee9ruOAggoKICAgIOWFs+mUruWPguaVsOivtOaYjjoKICAgICAgLSB0YXJnZXRfbW9kdWxlczog5a+55omA5pyJIGF0dGVudGlvbiArIEZGTiDmipXlvbHlsYLms6jlhaUgTG9SQQogICAgICAtIG1vZHVsZXNfdG9fc2F2ZT1bInNjb3JlIl06IOWIhuexu+WktCAoc2NvcmUpIOW/hemhu+WFqOmHj+iuree7g+W5tuS/neWtmO+8jAogICAgICAgIOWQpuWImemaj+acuuWIneWni+WMlueahOWIhuexu+WktOS4jeS8muiiq+aMgeS5heWMlgogICAgICAtIHRhc2tfdHlwZT1TRVFfQ0xTOiDlkYror4kgUEVGVCDov5nmmK/kuIDkuKrluo/liJfliIbnsbvku7vliqEKICAgICIiIgogICAgcmV0dXJuIExvcmFDb25maWcoCiAgICAgICAgcj1jb25maWcubG9yYV9yLAogICAgICAgIGxvcmFfYWxwaGE9Y29uZmlnLmxvcmFfYWxwaGEsCiAgICAgICAgdGFyZ2V0X21vZHVsZXM9Y29uZmlnLmxvcmFfdGFyZ2V0X21vZHVsZXMsCiAgICAgICAgbG9yYV9kcm9wb3V0PWNvbmZpZy5sb3JhX2Ryb3BvdXQsCiAgICAgICAgYmlhcz1jb25maWcubG9yYV9iaWFzLAogICAgICAgIHRhc2tfdHlwZT1UYXNrVHlwZS5TRVFfQ0xTLAogICAgICAgIG1vZHVsZXNfdG9fc2F2ZT1jb25maWcubG9yYV9tb2R1bGVzX3RvX3NhdmUsCiAgICApCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Yqg6L295YiG57G75qih5Z6LIChRTG9SQSkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX21vZGVsKGNvbmZpZzogTW9kZWxDb25maWcsIHRva2VuaXplcj1Ob25lKToKICAgICIiIgogICAg5Yqg6L29IFFMb1JBIOWIhuexu+aooeWei++8jOWujOaVtOa1geeoi++8mgogICAgICAxLiDkvb/nlKggQml0c0FuZEJ5dGVzQ29uZmlnIOi/m+ihjCA0LWJpdCDph4/ljJbliqDovb0KICAgICAgMi4g6YCa6L+HIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24g5pu/5o2i6K+t6KiA5bu65qih5aS05Li65YiG57G75aS0IChudW1fbGFiZWxzPTMpCiAgICAgIDMuIOS9v+eUqCBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nIOWHhuWkh+mHj+WMluaooeWeiwogICAgICA0LiDms6jlhaUgTG9SQSBhZGFwdGVyCgogICAgUmV0dXJuczoKICAgICAgICBQRUZUIOWMheijheWQjueahOaooeWei++8iOiLpSB1c2VfbG9yYT1UcnVl77yJ77yM5ZCm5YiZ5Y6f5aeL5qih5Z6L44CCCiAgICAiIiIKICAgIGJuYl9jb25maWcgPSBfYnVpbGRfYm5iX2NvbmZpZyhjb25maWcpCgogICAgIyBRd2VuMy41IOaYryBWTE0g5qih5Z6L77yMY29uZmlnIOacieW1jOWll+eahCB0ZXh0X2NvbmZpZ+OAggogICAgIyDpnIDopoHnoa7kv50gbnVtX2xhYmVscyDooqvmraPnoa7kvKDmkq3liLAgdGV4dF9jb25maWfjgIIKICAgIHRyeToKICAgICAgICBtb2RlbF9jb25maWcgPSBBdXRvQ29uZmlnLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgY29uZmlnLm1vZGVsX25hbWUsCiAgICAgICAgICAgIG51bV9sYWJlbHM9TlVNX0xBQkVMUywKICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgY2FjaGVfZGlyPWNvbmZpZy5jYWNoZV9kaXIsCiAgICAgICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9Y29uZmlnLmxvY2FsX2ZpbGVzX29ubHksCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb3mqKHlnovphY3nva7lpLHotKU6IHtleGN9XG4iCiAgICAgICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UoY29uZmlnLm1vZGVsX25hbWUpfSIKICAgICAgICApIGZyb20gZXhjCgogICAgIyBRd2VuMy41IFZMTSDnmoQgbnVtX2xhYmVscyDlj6/og73msqHmnInkvKDmkq3liLAgdGV4dF9jb25maWcKICAgIGlmIGhhc2F0dHIobW9kZWxfY29uZmlnLCAidGV4dF9jb25maWciKToKICAgICAgICBtb2RlbF9jb25maWcudGV4dF9jb25maWcubnVtX2xhYmVscyA9IE5VTV9MQUJFTFMKICAgICAgICBMT0dHRVIuaW5mbygi5bey5omL5Yqo5bCGIG51bV9sYWJlbHM9JXMg5Lyg5pKt5YiwIHRleHRfY29uZmlnIiwgTlVNX0xBQkVMUykKCiAgICBkdHlwZSA9IHRvcmNoLmZsb2F0MTYgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIHRvcmNoLmZsb2F0MzIKICAgIHRyeToKICAgICAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBjb25maWcubW9kZWxfbmFtZSwKICAgICAgICAgICAgY29uZmlnPW1vZGVsX2NvbmZpZywKICAgICAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1ibmJfY29uZmlnLAogICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICBjYWNoZV9kaXI9Y29uZmlnLmNhY2hlX2RpciwKICAgICAgICAgICAgbG9jYWxfZmlsZXNfb25seT1jb25maWcubG9jYWxfZmlsZXNfb25seSwKICAgICAgICAgICAgdG9yY2hfZHR5cGU9ZHR5cGUsCiAgICAgICAgKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgZiLliqDovb3mqKHlnovlpLHotKU6IHtleGN9XG4iCiAgICAgICAgICAgIGYie19kZXNjcmliZV9tb2RlbF9zb3VyY2UoY29uZmlnLm1vZGVsX25hbWUpfSIKICAgICAgICApIGZyb20gZXhjCgogICAgIyDlr7npvZAgcGFkX3Rva2VuX2lkCiAgICBpZiB0b2tlbml6ZXIgaXMgbm90IE5vbmUgYW5kIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwuY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsLmNvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgICAgIG1vZGVsLmNvbmZpZy50ZXh0X2NvbmZpZy5wYWRfdG9rZW5faWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCgogICAgaWYgbm90IGNvbmZpZy51c2VfbG9yYToKICAgICAgICByZXR1cm4gbW9kZWwKCiAgICAjIFFMb1JBIOeJueacieatpemqpDog5YeG5aSH6YeP5YyW5qih5Z6L5Lul6YCC6YWN6K6t57uD77yI5LuF5Zyo5a6e6ZmF6YeP5YyW5pe277yJCiAgICBpZiBjb25maWcubG9hZF9pbl80Yml0IGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIG1vZGVsID0gcHJlcGFyZV9tb2RlbF9mb3Jfa2JpdF90cmFpbmluZygKICAgICAgICAgICAgbW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPVRydWUKICAgICAgICApCgogICAgbG9yYV9jb25maWcgPSBfYnVpbGRfbG9yYV9jb25maWcoY29uZmlnKQogICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgbG9yYV9jb25maWcpCgogICAgIyDmiZPljbDlj6/orq3nu4Plj4LmlbDnu5/orqEKICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKCiAgICByZXR1cm4gbW9kZWwK",
    "modeling.py": "IiIiCuaooeWei+WumuS5ieaooeWdl+OAggoK5b2T5YmN54mI5pys5L2/55SoIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24gKyBRTG9SQe+8jArmqKHlnovnmoTliqDovb3lkozphY3nva7lt7Lnp7voh7MgaGYucHnjgILmraTmqKHlnZfkv53nlZnkuLrljaDkvY3vvIzkvpvmnKrmnaXmianlsZXkvb/nlKjjgIIKIiIiCg==",
    "dummy_data.py": "IiIiCuWBh+aVsOaNrueUn+aIkOaooeWdlyDigJQg55Sf5oiQIGR1bW15IHRyYWluLmNzdiDlkowgdGVzdC5jc3Yg55So5LqO5pys5Zyw5rWL6K+V44CCCgrkvb/nlKjmlrnms5XvvJoKICB1diBydW4gYXJlbmEtZHVtbXktZGF0YSAgICAgICAgICAgICAgICAgICAgIyDpu5jorqQgMTAwIOadoeiuree7gyArIDIwIOadoea1i+ivlQogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tbi10cmFpbiA1MCAtLW4tdGVzdCAxMAogIHV2IHJ1biBhcmVuYS1kdW1teS1kYXRhIC0tb3V0cHV0LWRpciAuL2RhdGEKCuS5n+WPr+S7peWcqCBQeXRob24g5Lit55u05o6l6LCD55So77yaCiAgZnJvbSBhcmVuYV9yYW5rZXIuZHVtbXlfZGF0YSBpbXBvcnQgZ2VuZXJhdGVfZHVtbXlfZGF0YQogIGdlbmVyYXRlX2R1bW15X2RhdGEobl90cmFpbj0xMDAsIG5fdGVzdD0yMCkKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHJhbmRvbQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgojIOWBh+aPkOmXruWIl+ihqApfUFJPTVBUUyA9IFsKICAgICJXaGF0IGlzIG1hY2hpbmUgbGVhcm5pbmc/IEV4cGxhaW4gaXQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV3JpdGUgYSBQeXRob24gZnVuY3Rpb24gdGhhdCBzb3J0cyBhIGxpc3QgdXNpbmcgbWVyZ2Ugc29ydC4iLAogICAgIkV4cGxhaW4gdGhlIHRoZW9yeSBvZiByZWxhdGl2aXR5IHRvIGEgMTAteWVhci1vbGQuIiwKICAgICJIb3cgZG9lcyBwaG90b3N5bnRoZXNpcyB3b3JrPyBQbGVhc2UgYmUgZGV0YWlsZWQuIiwKICAgICJXaGF0IGFyZSB0aGUgbWFpbiBkaWZmZXJlbmNlcyBiZXR3ZWVuIFRDUCBhbmQgVURQPyIsCiAgICAiU3VtbWFyaXplIHRoZSBwbG90IG9mIFJvbWVvIGFuZCBKdWxpZXQuIiwKICAgICJXcml0ZSBhIHBvZW0gYWJvdXQgdGhlIG9jZWFuIGF0IHN1bnNldC4iLAogICAgIkV4cGxhaW4gcXVhbnR1bSBlbnRhbmdsZW1lbnQgaW4gc2ltcGxlIHRlcm1zLiIsCiAgICAiV2hhdCBpcyB0aGUgY2FwaXRhbCBvZiBGcmFuY2UgYW5kIHdoYXQgaXMgaXQga25vd24gZm9yPyIsCiAgICAiSG93IGRvIG5ldXJhbCBuZXR3b3JrcyBsZWFybj8gRXhwbGFpbiBiYWNrcHJvcGFnYXRpb24uIiwKICAgICJDb21wYXJlIFB5dGhvbiBhbmQgUnVzdCBmb3Igc3lzdGVtcyBwcm9ncmFtbWluZy4iLAogICAgIldoYXQgYXJlIHRoZSBiZW5lZml0cyBvZiBtZWRpdGF0aW9uPyIsCiAgICAiRGVzY3JpYmUgaG93IGEgQ1BVIGV4ZWN1dGVzIGluc3RydWN0aW9ucy4iLAogICAgIldyaXRlIGEgU1FMIHF1ZXJ5IHRvIGZpbmQgZHVwbGljYXRlIGVtYWlscyBpbiBhIHRhYmxlLiIsCiAgICAiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIHN1cGVydmlzZWQgYW5kIHVuc3VwZXJ2aXNlZCBsZWFybmluZy4iLApdCgpfUkVTUE9OU0VfVEVNUExBVEVTX0EgPSBbCiAgICAiVGhhdCdzIGEgZ3JlYXQgcXVlc3Rpb24hIExldCBtZSBleHBsYWluLiB7dG9waWN9IGlzIGZ1bmRhbWVudGFsbHkgYWJvdXQgIgogICAgInVuZGVyc3RhbmRpbmcgcGF0dGVybnMgaW4gZGF0YS4gVGhlIGtleSBpbnNpZ2h0IGlzIHRoYXQgd2UgY2FuIHVzZSBtYXRoZW1hdGljYWwgIgogICAgIm1vZGVscyB0byBhcHByb3hpbWF0ZSBjb21wbGV4IHJlbGF0aW9uc2hpcHMuIEhlcmUncyBhIG1vcmUgZGV0YWlsZWQgYnJlYWtkb3duOiAiCiAgICAiRmlyc3QsIHdlIG5lZWQgdG8gY29sbGVjdCByZWxldmFudCBkYXRhLiBUaGVuLCB3ZSBwcmVwcm9jZXNzIGl0IHRvIHJlbW92ZSBub2lzZS4gIgogICAgIkZpbmFsbHksIHdlIHRyYWluIG91ciBtb2RlbCBhbmQgZXZhbHVhdGUgaXRzIHBlcmZvcm1hbmNlLiIsCiAgICAiU3VyZSEgSGVyZSdzIG15IHRha2Ugb24gdGhpcy4ge3RvcGljfSBpbnZvbHZlcyBzZXZlcmFsIGltcG9ydGFudCBjb25jZXB0cy4gIgogICAgIlRoZSBtb3N0IGZ1bmRhbWVudGFsIG9uZSBpcyB0aGF0IGxlYXJuaW5nIGhhcHBlbnMgdGhyb3VnaCBpdGVyYXRpdmUgb3B0aW1pemF0aW9uLiAiCiAgICAiV2Ugc3RhcnQgd2l0aCBhIHJhbmRvbSBndWVzcyBhbmQgZ3JhZHVhbGx5IGltcHJvdmUgaXQgYmFzZWQgb24gZmVlZGJhY2sgZnJvbSB0aGUgZGF0YS4iLAogICAgIkdyZWF0IHF1ZXN0aW9uLiBJbiBzaW1wbGUgdGVybXMsIHt0b3BpY30gaXMgbGlrZSB0ZWFjaGluZyBhIGNvbXB1dGVyIHRvIHJlY29nbml6ZSAiCiAgICAicGF0dGVybnMuIEltYWdpbmUgc2hvd2luZyBhIGNoaWxkIHRob3VzYW5kcyBvZiBwaWN0dXJlcyBvZiBjYXRzIGFuZCBkb2dzIC0gZXZlbnR1YWxseSAiCiAgICAidGhleSBsZWFybiB0byB0ZWxsIHRoZW0gYXBhcnQuIFRoYXQncyBlc3NlbnRpYWxseSB3aGF0IGhhcHBlbnMgaW4gdGhpcyBwcm9jZXNzLiIsCl0KCl9SRVNQT05TRV9URU1QTEFURVNfQiA9IFsKICAgICJUaGFua3MgZm9yIGFza2luZyEge3RvcGljfSBpcyBhY3R1YWxseSBzaW1wbGVyIHRoYW4gbW9zdCBwZW9wbGUgdGhpbmsuICIKICAgICJBdCBpdHMgY29yZSwgaXQncyBhYm91dCBmaW5kaW5nIHRoZSBiZXN0IGZ1bmN0aW9uIHRoYXQgbWFwcyBpbnB1dHMgdG8gb3V0cHV0cy4gIgogICAgIldlIGRvIHRoaXMgYnkgbWluaW1pemluZyBhIGxvc3MgZnVuY3Rpb24gdXNpbmcgZ3JhZGllbnQtYmFzZWQgb3B0aW1pemF0aW9uLiAiCiAgICAiVGhlIGJlYXV0eSBvZiB0aGlzIGFwcHJvYWNoIGlzIGl0cyBnZW5lcmFsaXR5IC0gaXQgd29ya3MgYWNyb3NzIG1hbnkgZG9tYWlucy4iLAogICAgIkxldCBtZSBicmVhayB0aGlzIGRvd24uIHt0b3BpY30gY2FuIGJlIHVuZGVyc3Rvb2QgdGhyb3VnaCBhIHNpbXBsZSBhbmFsb2d5LiAiCiAgICAiVGhpbmsgb2YgaXQgYXMgYSByZWNpcGUgLSB5b3UgaGF2ZSBpbmdyZWRpZW50cyAoZGF0YSksIGluc3RydWN0aW9ucyAoYWxnb3JpdGhtKSwgIgogICAgImFuZCBhIGZpbmFsIGRpc2ggKHByZWRpY3Rpb25zKS4gVGhlIHF1YWxpdHkgb2YgZWFjaCBpbmdyZWRpZW50IG1hdHRlcnMsIGJ1dCBzbyBkb2VzICIKICAgICJob3cgeW91IGNvbWJpbmUgdGhlbS4iLAogICAgIkknZCBiZSBoYXBweSB0byBleHBsYWluLiB7dG9waWN9IGlzIGEgZmFzY2luYXRpbmcgYXJlYS4gVGhlIGJhc2ljIGlkZWEgaXMgIgogICAgInRoYXQgd2UgY2FuIHVzZSBzdGF0aXN0aWNzIGFuZCBjb21wdXRhdGlvbiB0byBtYWtlIHByZWRpY3Rpb25zIGFib3V0IHRoZSB3b3JsZC4gIgogICAgIlRoZSBrZXkgY2hhbGxlbmdlIGlzIGdlbmVyYWxpemF0aW9uIC0gbWFraW5nIHN1cmUgb3VyIG1vZGVsIHdvcmtzIG9uIG5ldywgdW5zZWVuIGRhdGEuIiwKXQoKCmRlZiBnZW5lcmF0ZV9kdW1teV9kYXRhKAogICAgb3V0cHV0X2Rpcjogc3RyID0gIi4iLAogICAgbl90cmFpbjogaW50ID0gMTAwLAogICAgbl90ZXN0OiBpbnQgPSAyMCwKICAgIHNlZWQ6IGludCA9IDQyLAopIC0+IHR1cGxlW1BhdGgsIFBhdGhdOgogICAgIiIiCiAgICDnlJ/miJDnlKjkuo7mnKzlnLDmtYvor5XnmoTlgYfmlbDmja7jgIIKCiAgICDmoIfnrb7liIbluIPlpKfoh7TkuLrvvJpB6IOcIDQwJSwgQuiDnCA0MCUsIOW5s+WxgCAyMCXvvIjmqKHmi5/nnJ/lrp7mlbDmja7liIbluIPvvInjgIIKCiAgICBSZXR1cm5zOgogICAgICAgICh0cmFpbl9wYXRoLCB0ZXN0X3BhdGgpCiAgICAiIiIKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShzZWVkKQogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG91dCA9IFBhdGgob3V0cHV0X2RpcikKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyAtLS0tIOiuree7g+mbhiAtLS0tCiAgICB0cmFpbl9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdHJhaW4pOgogICAgICAgIHByb21wdCA9IF9QUk9NUFRTW2kgJSBsZW4oX1BST01QVFMpXQogICAgICAgIHRvcGljID0gcHJvbXB0LnNwbGl0KCI/IilbMF0uc3BsaXQoIi4iKVswXS5zdHJpcCgpCgogICAgICAgIHJlc3BfYSA9IHJhbmRvbS5jaG9pY2UoX1JFU1BPTlNFX1RFTVBMQVRFU19BKS5mb3JtYXQodG9waWM9dG9waWMpCiAgICAgICAgcmVzcF9iID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0IpLmZvcm1hdCh0b3BpYz10b3BpYykKCiAgICAgICAgbGFiZWwgPSBybmcuY2hvaWNlKFswLCAxLCAyXSwgcD1bMC40LCAwLjQsIDAuMl0pCiAgICAgICAgdHJhaW5fcm93cy5hcHBlbmQoewogICAgICAgICAgICAiaWQiOiAxMDAwMDAgKyBpLAogICAgICAgICAgICAibW9kZWxfYSI6IGYibW9kZWxfeF97cm5nLnJhbmRpbnQoMSwgNSl9IiwKICAgICAgICAgICAgIm1vZGVsX2IiOiBmIm1vZGVsX3lfe3JuZy5yYW5kaW50KDEsIDUpfSIsCiAgICAgICAgICAgICJwcm9tcHQiOiBwcm9tcHQsCiAgICAgICAgICAgICJyZXNwb25zZV9hIjogcmVzcF9hLAogICAgICAgICAgICAicmVzcG9uc2VfYiI6IHJlc3BfYiwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9hIjogMSBpZiBsYWJlbCA9PSAwIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl9tb2RlbF9iIjogMSBpZiBsYWJlbCA9PSAxIGVsc2UgMCwKICAgICAgICAgICAgIndpbm5lcl90aWUiOiAxIGlmIGxhYmVsID09IDIgZWxzZSAwLAogICAgICAgIH0pCgogICAgdHJhaW5fcGF0aCA9IG91dCAvICJ0cmFpbi5jc3YiCiAgICBwZC5EYXRhRnJhbWUodHJhaW5fcm93cykudG9fY3N2KHRyYWluX3BhdGgsIGluZGV4PUZhbHNlKQoKICAgICMgLS0tLSDmtYvor5Xpm4YgLS0tLQogICAgdGVzdF9yb3dzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG5fdGVzdCk6CiAgICAgICAgcHJvbXB0ID0gX1BST01QVFNbaSAlIGxlbihfUFJPTVBUUyldCiAgICAgICAgdG9waWMgPSBwcm9tcHQuc3BsaXQoIj8iKVswXS5zcGxpdCgiLiIpWzBdLnN0cmlwKCkKCiAgICAgICAgcmVzcF9hID0gcmFuZG9tLmNob2ljZShfUkVTUE9OU0VfVEVNUExBVEVTX0EpLmZvcm1hdCh0b3BpYz10b3BpYykKICAgICAgICByZXNwX2IgPSByYW5kb20uY2hvaWNlKF9SRVNQT05TRV9URU1QTEFURVNfQikuZm9ybWF0KHRvcGljPXRvcGljKQoKICAgICAgICB0ZXN0X3Jvd3MuYXBwZW5kKHsKICAgICAgICAgICAgImlkIjogMjAwMDAwICsgaSwKICAgICAgICAgICAgInByb21wdCI6IHByb21wdCwKICAgICAgICAgICAgInJlc3BvbnNlX2EiOiByZXNwX2EsCiAgICAgICAgICAgICJyZXNwb25zZV9iIjogcmVzcF9iLAogICAgICAgIH0pCgogICAgdGVzdF9wYXRoID0gb3V0IC8gInRlc3QuY3N2IgogICAgcGQuRGF0YUZyYW1lKHRlc3Rfcm93cykudG9fY3N2KHRlc3RfcGF0aCwgaW5kZXg9RmFsc2UpCgogICAgcHJpbnQoZiLlt7LnlJ/miJDorq3nu4Ppm4Y6IHt0cmFpbl9wYXRofSAoe25fdHJhaW59IOadoSkiKQogICAgcHJpbnQoZiLlt7LnlJ/miJDmtYvor5Xpm4Y6IHt0ZXN0X3BhdGh9ICh7bl90ZXN0fSDmnaEpIikKICAgIHJldHVybiB0cmFpbl9wYXRoLCB0ZXN0X3BhdGgKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj0i55Sf5oiQ5YGH5pWw5o2u55So5LqO5pys5Zyw5rWL6K+V44CCIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSIuIiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye655uu5b2VIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10cmFpbiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMCwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6K6t57uD6ZuG6KGM5pWwIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbi10ZXN0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iua1i+ivlembhuihjOaVsCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZ2VuZXJhdGVfZHVtbXlfZGF0YSgKICAgICAgICBvdXRwdXRfZGlyPWFyZ3Mub3V0cHV0X2RpciwKICAgICAgICBuX3RyYWluPWFyZ3Mubl90cmFpbiwKICAgICAgICBuX3Rlc3Q9YXJncy5uX3Rlc3QsCiAgICAgICAgc2VlZD1hcmdzLnNlZWQsCiAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "train.py": "IiIiCuiuree7g+iEmuacrCDigJQg5L2/55SoIEh1Z2dpbmdGYWNlIFRyYWluZXIg6L+b6KGMIFFMb1JBIOW+ruiwg+OAggoK5a6M5pW05rWB56iL77yaCiAgMS4g5Yqg6L296YWN572uICjpu5jorqTlgLwgKyBZQU1MIOimhuebliArIENMSSDopobnm5YpCiAgMi4g55Sf5oiQ5oiW5Yqg6L296K6t57uD5pWw5o2uCiAgMy4gVG9rZW5pemF0aW9uIChhcHBseV9jaGF0X3RlbXBsYXRlKQogIDQuIOWKoOi9vSBRd2VuMy41LTAuOEIgKyA0LWJpdCDph4/ljJYgKyBMb1JBCiAgNS4g5L2/55SoIFRyYWluZXIg6K6t57uDCiAgNi4g5L+d5a2YIGFkYXB0ZXIgKyB0b2tlbml6ZXIgKyDphY3nva4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHJhbmRvbQppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IERhdGFDb2xsYXRvcldpdGhQYWRkaW5nLCBUcmFpbmVyLCBUcmFpbmluZ0FyZ3VtZW50cwoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBBcHBDb25maWcsIGxvYWRfY29uZmlnCmZyb20gYXJlbmFfcmFua2VyLmRhdGEgaW1wb3J0ICgKICAgIGJ1aWxkX2RhdGFzZXQsCiAgICBjb21wdXRlX21ldHJpY3MsCiAgICBsb2FkX2FuZF9wcmVwcm9jZXNzLAogICAgc3BsaXRfdHJhaW5fdmFsaWQsCikKZnJvbSBhcmVuYV9yYW5rZXIuaGYgaW1wb3J0IGxvYWRfbW9kZWwsIGxvYWRfdG9rZW5pemVyCgpMT0dHRVIgPSBsb2dnaW5nLmdldExvZ2dlcigiYXJlbmFfcmFua2VyLnRyYWluIikKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDTEkg5Y+C5pWw6Kej5p6QCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgcGFyc2VfYXJncygpIC0+IGFyZ3BhcnNlLk5hbWVzcGFjZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKAogICAgICAgIGRlc2NyaXB0aW9uPSJRTG9SQSBmaW5lLXR1bmUgUXdlbjMuNS0wLjhCIGZvciBBcmVuYSBwcmVmZXJlbmNlIHByZWRpY3Rpb24uIgogICAgKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCB0eXBlPXN0ciwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJZQU1MIOmFjee9ruaWh+S7tui3r+W+hCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEtZGlyIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Ii4iLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0cmFpbi5jc3YgLyB0ZXN0LmNzdiDmiYDlnKjnm67lvZUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb2RlbC1uYW1lIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5Z+65bqn5qih5Z6L5ZCN56ew5oiW5pys5Zyw6Lev5b6EIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PU5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iuiuree7g+S6p+eJqei+k+WHuuebruW9lSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heC1sZW5ndGgiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ0b2tlbml6ZXIg5pyA5aSn6ZW/5bqmIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6K6t57uDIGVwb2NoIOaVsCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJwZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncmFkLWFjY3VtLXN0ZXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5qKv5bqm57Sv56ev5q2l5pWwIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGVhcm5pbmctcmF0ZSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i5a2m5Lmg546HIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td2FybXVwLXJhdGlvIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJ3YXJtdXAg5Y2g5oC76K6t57uD5q2l5pWw55qE5q+U5L6LIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0td2FybXVwLXN0ZXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0id2FybXVwIOe7neWvueatpeaVsO+8m+iuvue9ruWQjuimhuebliB3YXJtdXBfcmF0aW8iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jYWNoZS1kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD1Ob25lKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sb2NhbC1maWxlcy1vbmx5IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGlzYWJsZS1sb3JhIiwgZGVzdD0idXNlX2xvcmEiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbG9yYS1yIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbG9yYS1hbHBoYSIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW5vLTRiaXQiLCBkZXN0PSJsb2FkX2luXzRiaXQiLCBhY3Rpb249InN0b3JlX2ZhbHNlIikKICAgIHBhcnNlci5zZXRfZGVmYXVsdHModXNlX2xvcmE9Tm9uZSwgbG9hZF9pbl80Yml0PU5vbmUpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCmRlZiBhcHBseV9vdmVycmlkZXMoY29uZmlnOiBBcHBDb25maWcsIGFyZ3M6IGFyZ3BhcnNlLk5hbWVzcGFjZSkgLT4gQXBwQ29uZmlnOgogICAgIiIi5bCGIENMSSDlj4LmlbDopobnm5bliLDphY3nva7kuK3jgIIiIiIKICAgIGlmIGFyZ3MubW9kZWxfbmFtZToKICAgICAgICBjb25maWcubW9kZWwubW9kZWxfbmFtZSA9IGFyZ3MubW9kZWxfbmFtZQogICAgaWYgYXJncy5vdXRwdXRfZGlyOgogICAgICAgIGNvbmZpZy50cmFpbmluZy5vdXRwdXRfZGlyID0gYXJncy5vdXRwdXRfZGlyCiAgICBpZiBhcmdzLm1heF9sZW5ndGggaXMgbm90IE5vbmU6CiAgICAgICAgY29uZmlnLm1vZGVsLm1heF9sZW5ndGggPSBhcmdzLm1heF9sZW5ndGgKICAgIGlmIGFyZ3MuZXBvY2hzIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy50cmFpbmluZy5udW1fdHJhaW5fZXBvY2hzID0gYXJncy5lcG9jaHMKICAgIGlmIGFyZ3MuYmF0Y2hfc2l6ZSBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcudHJhaW5pbmcucGVyX2RldmljZV90cmFpbl9iYXRjaF9zaXplID0gYXJncy5iYXRjaF9zaXplCiAgICBpZiBhcmdzLmdyYWRfYWNjdW1fc3RlcHMgaXMgbm90IE5vbmU6CiAgICAgICAgY29uZmlnLnRyYWluaW5nLmdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcyA9IGFyZ3MuZ3JhZF9hY2N1bV9zdGVwcwogICAgaWYgYXJncy5sZWFybmluZ19yYXRlIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy50cmFpbmluZy5sZWFybmluZ19yYXRlID0gYXJncy5sZWFybmluZ19yYXRlCiAgICBpZiBhcmdzLndhcm11cF9yYXRpbyBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcudHJhaW5pbmcud2FybXVwX3JhdGlvID0gYXJncy53YXJtdXBfcmF0aW8KICAgIGlmIGFyZ3Mud2FybXVwX3N0ZXBzIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy50cmFpbmluZy53YXJtdXBfc3RlcHMgPSBhcmdzLndhcm11cF9zdGVwcwogICAgaWYgYXJncy5jYWNoZV9kaXIgaXMgbm90IE5vbmU6CiAgICAgICAgY29uZmlnLm1vZGVsLmNhY2hlX2RpciA9IGFyZ3MuY2FjaGVfZGlyCiAgICBpZiBhcmdzLmxvY2FsX2ZpbGVzX29ubHk6CiAgICAgICAgY29uZmlnLm1vZGVsLmxvY2FsX2ZpbGVzX29ubHkgPSBUcnVlCiAgICBpZiBhcmdzLnVzZV9sb3JhIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC51c2VfbG9yYSA9IGFyZ3MudXNlX2xvcmEKICAgIGlmIGFyZ3MubG9yYV9yIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC5sb3JhX3IgPSBhcmdzLmxvcmFfcgogICAgaWYgYXJncy5sb3JhX2FscGhhIGlzIG5vdCBOb25lOgogICAgICAgIGNvbmZpZy5tb2RlbC5sb3JhX2FscGhhID0gYXJncy5sb3JhX2FscGhhCiAgICBpZiBhcmdzLmxvYWRfaW5fNGJpdCBpcyBub3QgTm9uZToKICAgICAgICBjb25maWcubW9kZWwubG9hZF9pbl80Yml0ID0gYXJncy5sb2FkX2luXzRiaXQKICAgIHJldHVybiBjb25maWcKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDlt6Xlhbflh73mlbAKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBzZXRfc2VlZChzZWVkOiBpbnQpIC0+IE5vbmU6CiAgICByYW5kb20uc2VlZChzZWVkKQogICAgbnAucmFuZG9tLnNlZWQoc2VlZCkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQoKCmRlZiBzZXR1cF9sb2dnaW5nKCkgLT4gTm9uZToKICAgIGxvZ2dpbmcuYmFzaWNDb25maWcoCiAgICAgICAgbGV2ZWw9bG9nZ2luZy5JTkZPLAogICAgICAgIGZvcm1hdD0iJShhc2N0aW1lKXMgfCAlKGxldmVsbmFtZSlzIHwgJShtZXNzYWdlKXMiLAogICAgICAgIGRhdGVmbXQ9IiVIOiVNOiVTIiwKICAgICkKCgpkZWYgZm9ybWF0X3NlY29uZHMoc2Vjb25kczogZmxvYXQpIC0+IHN0cjoKICAgIHRvdGFsID0gbWF4KGludChzZWNvbmRzKSwgMCkKICAgIG0sIHMgPSBkaXZtb2QodG90YWwsIDYwKQogICAgaCwgbSA9IGRpdm1vZChtLCA2MCkKICAgIGlmIGggPiAwOgogICAgICAgIHJldHVybiBmIntofWgge219bSB7c31zIgogICAgaWYgbSA+IDA6CiAgICAgICAgcmV0dXJuIGYie219bSB7c31zIgogICAgcmV0dXJuIGYie3N9cyIKCgpkZWYgZGVzY3JpYmVfZGV2aWNlKGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBzdHI6CiAgICBpZiBkZXZpY2UudHlwZSAhPSAiY3VkYSI6CiAgICAgICAgcmV0dXJuICJDUFUiCiAgICBuYW1lID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoZGV2aWNlKQogICAgbWVtID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoZGV2aWNlKS50b3RhbF9tZW1vcnkgLyAxMDI0KiozCiAgICByZXR1cm4gZiJ7bmFtZX0gKHttZW06LjFmfSBHQikiCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5p6E5bu6IFRyYWluaW5nQXJndW1lbnRzCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgYnVpbGRfdHJhaW5pbmdfYXJncyhjb25maWc6IEFwcENvbmZpZykgLT4gVHJhaW5pbmdBcmd1bWVudHM6CiAgICAiIiLku44gQXBwQ29uZmlnIOaehOW7uiBIdWdnaW5nRmFjZSBUcmFpbmluZ0FyZ3VtZW50c+OAgiIiIgogICAgdGMgPSBjb25maWcudHJhaW5pbmcKICAgIGhhc19jdWRhID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQoKICAgICMgQ1BVIOaooeW8j+S4i+iHquWKqOiwg+aVtOS4jeWFvOWuueeahOWPguaVsAogICAgb3B0aW0gPSB0Yy5vcHRpbQogICAgZnAxNiA9IHRjLmZwMTYKICAgIGJmMTYgPSB0Yy5iZjE2CiAgICBpZiBub3QgaGFzX2N1ZGE6CiAgICAgICAgaWYgb3B0aW0uc3RhcnRzd2l0aCgicGFnZWRfIik6CiAgICAgICAgICAgIG9wdGltID0gImFkYW13X3RvcmNoIgogICAgICAgICAgICBMT0dHRVIud2FybmluZygiQ1BVIOaooeW8jzog5LyY5YyW5Zmo5LuOICVzIOWbnumAgOS4uiBhZGFtd190b3JjaCIsIHRjLm9wdGltKQogICAgICAgIGZwMTYgPSBGYWxzZQogICAgICAgIGJmMTYgPSBGYWxzZQogICAgICAgIExPR0dFUi53YXJuaW5nKCJDUFUg5qih5byPOiDlt7LnpoHnlKggZnAxNi9iZjE2IOa3t+WQiOeyvuW6piIpCgogICAgcmV0dXJuIFRyYWluaW5nQXJndW1lbnRzKAogICAgICAgIG91dHB1dF9kaXI9dGMub3V0cHV0X2RpciwKICAgICAgICBsZWFybmluZ19yYXRlPXRjLmxlYXJuaW5nX3JhdGUsCiAgICAgICAgd2VpZ2h0X2RlY2F5PXRjLndlaWdodF9kZWNheSwKICAgICAgICBwZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemU9dGMucGVyX2RldmljZV90cmFpbl9iYXRjaF9zaXplLAogICAgICAgIHBlcl9kZXZpY2VfZXZhbF9iYXRjaF9zaXplPXRjLnBlcl9kZXZpY2VfZXZhbF9iYXRjaF9zaXplLAogICAgICAgIGdyYWRpZW50X2FjY3VtdWxhdGlvbl9zdGVwcz10Yy5ncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMsCiAgICAgICAgbnVtX3RyYWluX2Vwb2Nocz10Yy5udW1fdHJhaW5fZXBvY2hzLAogICAgICAgIHdhcm11cF9yYXRpbz10Yy53YXJtdXBfcmF0aW8sCiAgICAgICAgd2FybXVwX3N0ZXBzPXRjLndhcm11cF9zdGVwcywKICAgICAgICBscl9zY2hlZHVsZXJfdHlwZT10Yy5scl9zY2hlZHVsZXJfdHlwZSwKICAgICAgICBvcHRpbT1vcHRpbSwKICAgICAgICBmcDE2PWZwMTYsCiAgICAgICAgYmYxNj1iZjE2LAogICAgICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc9dGMuZ3JhZGllbnRfY2hlY2twb2ludGluZywKICAgICAgICBsb2dnaW5nX3N0ZXBzPXRjLmxvZ2dpbmdfc3RlcHMsCiAgICAgICAgZXZhbF9zdHJhdGVneT10Yy5ldmFsX3N0cmF0ZWd5LAogICAgICAgIHNhdmVfc3RyYXRlZ3k9dGMuc2F2ZV9zdHJhdGVneSwKICAgICAgICBzYXZlX3RvdGFsX2xpbWl0PXRjLnNhdmVfdG90YWxfbGltaXQsCiAgICAgICAgbG9hZF9iZXN0X21vZGVsX2F0X2VuZD10Yy5sb2FkX2Jlc3RfbW9kZWxfYXRfZW5kLAogICAgICAgIG1ldHJpY19mb3JfYmVzdF9tb2RlbD10Yy5tZXRyaWNfZm9yX2Jlc3RfbW9kZWwsCiAgICAgICAgZ3JlYXRlcl9pc19iZXR0ZXI9dGMuZ3JlYXRlcl9pc19iZXR0ZXIsCiAgICAgICAgc2VlZD10Yy5zZWVkLAogICAgICAgIHJlcG9ydF90bz10Yy5yZXBvcnRfdG8sCiAgICAgICAgZGF0YWxvYWRlcl9udW1fd29ya2Vycz10Yy5kYXRhbG9hZGVyX251bV93b3JrZXJzLAogICAgICAgICMgUEVGVCArIGdyYWRpZW50IGNoZWNrcG9pbnRpbmcg5YW85a655oCnCiAgICAgICAgZ3JhZGllbnRfY2hlY2twb2ludGluZ19rd2FyZ3M9eyJ1c2VfcmVlbnRyYW50IjogRmFsc2V9CiAgICAgICAgaWYgdGMuZ3JhZGllbnRfY2hlY2twb2ludGluZwogICAgICAgIGVsc2UgTm9uZSwKICAgICkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDml6Xlv5fmkZjopoEKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2dfcnVuX3N1bW1hcnkoCiAgICBjb25maWc6IEFwcENvbmZpZywKICAgIGRldmljZTogdG9yY2guZGV2aWNlLAogICAgdHJhaW5fc2l6ZTogaW50LAogICAgdmFsaWRfc2l6ZTogaW50LAopIC0+IE5vbmU6CiAgICBtYyA9IGNvbmZpZy5tb2RlbAogICAgdGMgPSBjb25maWcudHJhaW5pbmcKICAgIExPR0dFUi5pbmZvKCI9IiAqIDYwKQogICAgTE9HR0VSLmluZm8oIuiuree7g+WQr+WKqCIpCiAgICBMT0dHRVIuaW5mbygi6K6+5aSHOiAlcyIsIGRlc2NyaWJlX2RldmljZShkZXZpY2UpKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIuaooeWeizogJXMgfCBtYXhfbGVuZ3RoPSVzIHwgNGJpdD0lcyB8IExvUkE9JXMiLAogICAgICAgIG1jLm1vZGVsX25hbWUsIG1jLm1heF9sZW5ndGgsCiAgICAgICAgIm9uIiBpZiBtYy5sb2FkX2luXzRiaXQgZWxzZSAib2ZmIiwKICAgICAgICAib24iIGlmIG1jLnVzZV9sb3JhIGVsc2UgIm9mZiIsCiAgICApCiAgICBpZiBtYy51c2VfbG9yYToKICAgICAgICBMT0dHRVIuaW5mbygKICAgICAgICAgICAgIkxvUkEg6YWN572uOiByPSVzLCBhbHBoYT0lcywgZHJvcG91dD0lLjNmLCBtb2R1bGVzPSVzLCBtb2R1bGVzX3RvX3NhdmU9JXMiLAogICAgICAgICAgICBtYy5sb3JhX3IsIG1jLmxvcmFfYWxwaGEsIG1jLmxvcmFfZHJvcG91dCwKICAgICAgICAgICAgIiwiLmpvaW4obWMubG9yYV90YXJnZXRfbW9kdWxlcyksCiAgICAgICAgICAgICIsIi5qb2luKG1jLmxvcmFfbW9kdWxlc190b19zYXZlKSwKICAgICAgICApCiAgICBMT0dHRVIuaW5mbygKICAgICAgICAi5pWw5o2u6ZuGOiB0cmFpbj0lcywgdmFsaWQ9JXMiLAogICAgICAgIHRyYWluX3NpemUsIHZhbGlkX3NpemUsCiAgICApCiAgICBMT0dHRVIuaW5mbygKICAgICAgICAi6K6t57uD5Y+C5pWwOiBscj0lLjFlLCBlcG9jaHM9JXMsIGJhdGNoPSVzLCBncmFkX2FjY3VtPSVzLCBzY2hlZHVsZXI9JXMsIHdhcm11cD0lcyIsCiAgICAgICAgdGMubGVhcm5pbmdfcmF0ZSwgdGMubnVtX3RyYWluX2Vwb2NocywKICAgICAgICB0Yy5wZXJfZGV2aWNlX3RyYWluX2JhdGNoX3NpemUsCiAgICAgICAgdGMuZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzLAogICAgICAgIHRjLmxyX3NjaGVkdWxlcl90eXBlLAogICAgICAgIGYie3RjLndhcm11cF9zdGVwc30gc3RlcHMiIGlmIHRjLndhcm11cF9zdGVwcyA+IDAgZWxzZSBmInt0Yy53YXJtdXBfcmF0aW86LjIlfSIsCiAgICApCiAgICBMT0dHRVIuaW5mbygi6L6T5Ye655uu5b2VOiAlcyIsIHRjLm91dHB1dF9kaXIpCiAgICBMT0dHRVIuaW5mbygiPSIgKiA2MCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDkv53lrZjorq3nu4PkuqfniakKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBzYXZlX2FydGlmYWN0cygKICAgIG91dHB1dF9kaXI6IFBhdGgsCiAgICBtb2RlbCwKICAgIHRva2VuaXplciwKICAgIGNvbmZpZzogQXBwQ29uZmlnLAogICAgbWV0cmljczogZGljdCB8IE5vbmUgPSBOb25lLAopIC0+IE5vbmU6CiAgICAiIiIKICAgIOS/neWtmOiuree7g+S6p+eJqe+8mgogICAgICAtIGFkYXB0ZXIg5p2D6YeNIChMb1JBKSArIOWIhuexu+WktAogICAgICAtIHRva2VuaXplcgogICAgICAtIOmFjee9ruaWh+S7tgogICAgICAtIOacgOS9s+aMh+aghyAo5aaC5pyJKQogICAgIiIiCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICAjIOS/neWtmCBhZGFwdGVyIChQRUZUIOaooeWeiykg5oiW5a6M5pW05qih5Z6LCiAgICBtb2RlbC5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpciAvICJtb2RlbCIpCiAgICB0b2tlbml6ZXIuc2F2ZV9wcmV0cmFpbmVkKG91dHB1dF9kaXIgLyAidG9rZW5pemVyIikKICAgIGNvbmZpZy5zYXZlKG91dHB1dF9kaXIgLyAiY29uZmlnLnlhbWwiKQoKICAgIGlmIG1ldHJpY3M6CiAgICAgICAgKG91dHB1dF9kaXIgLyAibWV0cmljcy5qc29uIikud3JpdGVfdGV4dCgKICAgICAgICAgICAganNvbi5kdW1wcyhtZXRyaWNzLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlKSwKICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICAgICApCgogICAgTE9HR0VSLmluZm8oIuiuree7g+S6p+eJqeW3suS/neWtmOWIsDogJXMiLCBvdXRwdXRfZGlyKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOS4u+WHveaVsAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgc2V0dXBfbG9nZ2luZygpCiAgICBhcmdzID0gcGFyc2VfYXJncygpCiAgICBjb25maWcgPSBhcHBseV9vdmVycmlkZXMobG9hZF9jb25maWcoYXJncy5jb25maWcpLCBhcmdzKQogICAgc2V0X3NlZWQoY29uZmlnLnRyYWluaW5nLnNlZWQpCgogICAgZGF0YV9kaXIgPSBQYXRoKGFyZ3MuZGF0YV9kaXIpCiAgICBvdXRwdXRfZGlyID0gUGF0aChjb25maWcudHJhaW5pbmcub3V0cHV0X2RpcikKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgc3RhcnRlZF9hdCA9IHRpbWUucGVyZl9jb3VudGVyKCkKCiAgICAjIC0tLS0gMS4g5Yqg6L295pWw5o2uIC0tLS0KICAgIExPR0dFUi5pbmZvKCLliqDovb3orq3nu4PmlbDmja46ICVzIiwgZGF0YV9kaXIgLyBjb25maWcuZGF0YS50cmFpbl9wYXRoKQogICAgdHJhaW5fZGYgPSBsb2FkX2FuZF9wcmVwcm9jZXNzKAogICAgICAgIHN0cihkYXRhX2RpciAvIGNvbmZpZy5kYXRhLnRyYWluX3BhdGgpLAogICAgICAgIG1heF9jaGFycz1jb25maWcuZGF0YS50ZXh0X21heF9jaGFycywKICAgICAgICBpc190cmFpbj1UcnVlLAogICAgKQogICAgdHJhaW5fc3BsaXQsIHZhbGlkX3NwbGl0ID0gc3BsaXRfdHJhaW5fdmFsaWQodHJhaW5fZGYsIGNvbmZpZy5kYXRhKQogICAgTE9HR0VSLmluZm8oIuiuree7g+mbhjogJXMg5p2hLCDpqozor4Hpm4Y6ICVzIOadoSIsIGxlbih0cmFpbl9zcGxpdCksIGxlbih2YWxpZF9zcGxpdCkpCgogICAgIyAtLS0tIDIuIOWKoOi9vSB0b2tlbml6ZXIgLS0tLQogICAgTE9HR0VSLmluZm8oIuWKoOi9vSB0b2tlbml6ZXI6ICVzIiwgY29uZmlnLm1vZGVsLm1vZGVsX25hbWUpCiAgICB0b2tlbml6ZXIgPSBsb2FkX3Rva2VuaXplcihjb25maWcubW9kZWwpCgogICAgIyAtLS0tIDMuIFRva2VuaXplIOaVsOaNrumbhiAtLS0tCiAgICBMT0dHRVIuaW5mbygiVG9rZW5pemluZyDmlbDmja7pm4YgKG1heF9sZW5ndGg9JXMpLi4uIiwgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgpCiAgICB0cmFpbl9kYXRhc2V0ID0gYnVpbGRfZGF0YXNldCgKICAgICAgICB0cmFpbl9zcGxpdCwKICAgICAgICB0b2tlbml6ZXIsCiAgICAgICAgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsCiAgICAgICAgaXNfdHJhaW49VHJ1ZSwKICAgICAgICBpbmNsdWRlX3N3YXA9Y29uZmlnLmRhdGEuaW5jbHVkZV9zd2FwX3RyYWluLAogICAgKQogICAgdmFsaWRfZGF0YXNldCA9IGJ1aWxkX2RhdGFzZXQoCiAgICAgICAgdmFsaWRfc3BsaXQsCiAgICAgICAgdG9rZW5pemVyLAogICAgICAgIGNvbmZpZy5tb2RlbC5tYXhfbGVuZ3RoLAogICAgICAgIGlzX3RyYWluPVRydWUsCiAgICApCiAgICBMT0dHRVIuaW5mbygKICAgICAgICAiVG9rZW5pemF0aW9uIOWujOaIkDogdHJhaW49JXMsIHZhbGlkPSVzIiwKICAgICAgICBsZW4odHJhaW5fZGF0YXNldCksIGxlbih2YWxpZF9kYXRhc2V0KSwKICAgICkKCiAgICAjIC0tLS0gNC4g5Yqg6L295qih5Z6LIC0tLS0KICAgIExPR0dFUi5pbmZvKCLliqDovb3mqKHlnosgKFFMb1JBKTogJXMiLCBjb25maWcubW9kZWwubW9kZWxfbmFtZSkKICAgIG1vZGVsID0gbG9hZF9tb2RlbChjb25maWcubW9kZWwsIHRva2VuaXplcj10b2tlbml6ZXIpCgogICAgbG9nX3J1bl9zdW1tYXJ5KGNvbmZpZywgZGV2aWNlLCBsZW4odHJhaW5fZGF0YXNldCksIGxlbih2YWxpZF9kYXRhc2V0KSkKCiAgICAjIC0tLS0gNS4g5p6E5bu6IFRyYWluZXIgLS0tLQogICAgdHJhaW5pbmdfYXJncyA9IGJ1aWxkX3RyYWluaW5nX2FyZ3MoY29uZmlnKQogICAgZGF0YV9jb2xsYXRvciA9IERhdGFDb2xsYXRvcldpdGhQYWRkaW5nKAogICAgICAgIHRva2VuaXplcj10b2tlbml6ZXIsCiAgICAgICAgcGFkZGluZz1UcnVlLAogICAgKQoKICAgIHRyYWluZXIgPSBUcmFpbmVyKAogICAgICAgIG1vZGVsPW1vZGVsLAogICAgICAgIGFyZ3M9dHJhaW5pbmdfYXJncywKICAgICAgICB0cmFpbl9kYXRhc2V0PXRyYWluX2RhdGFzZXQsCiAgICAgICAgZXZhbF9kYXRhc2V0PXZhbGlkX2RhdGFzZXQsCiAgICAgICAgZGF0YV9jb2xsYXRvcj1kYXRhX2NvbGxhdG9yLAogICAgICAgIGNvbXB1dGVfbWV0cmljcz1jb21wdXRlX21ldHJpY3MsCiAgICApCgogICAgIyAtLS0tIDYuIOiuree7gyAtLS0tCiAgICBMT0dHRVIuaW5mbygi5byA5aeL6K6t57uDLi4uIikKICAgIHRyYWluX3Jlc3VsdCA9IHRyYWluZXIudHJhaW4oKQoKICAgICMgLS0tLSA3LiDor4TkvLAgLS0tLQogICAgTE9HR0VSLmluZm8oIuiuree7g+WujOaIkO+8jOi/kOihjOacgOe7iOivhOS8sC4uLiIpCiAgICBldmFsX3Jlc3VsdCA9IHRyYWluZXIuZXZhbHVhdGUoKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIuacgOe7iOivhOS8sDogbG9nX2xvc3M9JS40ZiwgYWNjdXJhY3k9JS40ZiIsCiAgICAgICAgZXZhbF9yZXN1bHQuZ2V0KCJldmFsX2xvZ19sb3NzIiwgZmxvYXQoIm5hbiIpKSwKICAgICAgICBldmFsX3Jlc3VsdC5nZXQoImV2YWxfYWNjdXJhY3kiLCBmbG9hdCgibmFuIikpLAogICAgKQoKICAgICMgLS0tLSA4LiDkv53lrZggLS0tLQogICAgbWV0cmljcyA9IHsKICAgICAgICAidHJhaW5fbG9zcyI6IHRyYWluX3Jlc3VsdC50cmFpbmluZ19sb3NzLAogICAgICAgICJldmFsX2xvZ19sb3NzIjogZXZhbF9yZXN1bHQuZ2V0KCJldmFsX2xvZ19sb3NzIiksCiAgICAgICAgImV2YWxfYWNjdXJhY3kiOiBldmFsX3Jlc3VsdC5nZXQoImV2YWxfYWNjdXJhY3kiKSwKICAgIH0KICAgIHNhdmVfYXJ0aWZhY3RzKG91dHB1dF9kaXIsIG1vZGVsLCB0b2tlbml6ZXIsIGNvbmZpZywgbWV0cmljcykKCiAgICBlbGFwc2VkID0gZm9ybWF0X3NlY29uZHModGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWRfYXQpCiAgICBMT0dHRVIuaW5mbygi5YWo6YOo5a6M5oiQ77yM5oC76ICX5pe2OiAlcyIsIGVsYXBzZWQpCiAgICBwcmludChqc29uLmR1bXBzKG1ldHJpY3MsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
    "predict.py": "IiIiCuaOqOeQhuiEmuacrCDigJQg5Yqg6L296K6t57uD5aW955qEIFFMb1JBIGFkYXB0ZXLvvIzlr7nmtYvor5Xpm4bnlJ/miJAgc3VibWlzc2lvbi5jc3bjgIIKCua1geeoi++8mgogIDEuIOmHjeaWsOWKoOi9veWfuuW6p+aooeWeiyAoNC1iaXQg6YeP5YyWKQogIDIuIOWKoOi9veS/neWtmOeahCBMb1JBIGFkYXB0ZXIgKyDliIbnsbvlpLQKICAzLiDpgY3ljobmtYvor5Xpm4bvvIznlJ/miJDkuInliIbnsbvmpoLnjocKICA0LiDmpoLnjofoo4HliaogKyDlvZLkuIDljJblkI7lhpnlhaUgc3VibWlzc2lvbi5jc3YKIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGxvZ2dpbmcKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCmZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCmZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoCiAgICBBdXRvQ29uZmlnLAogICAgQXV0b01vZGVsRm9yU2VxdWVuY2VDbGFzc2lmaWNhdGlvbiwKICAgIEF1dG9Ub2tlbml6ZXIsCiAgICBCaXRzQW5kQnl0ZXNDb25maWcsCiAgICBEYXRhQ29sbGF0b3JXaXRoUGFkZGluZywKKQoKZnJvbSBhcmVuYV9yYW5rZXIuY29uZmlnIGltcG9ydCBJRF9UT19MQUJFTCwgTlVNX0xBQkVMUywgbG9hZF9jb25maWcKZnJvbSBhcmVuYV9yYW5rZXIuZGF0YSBpbXBvcnQgYnVpbGRfZGF0YXNldCwgbG9hZF9hbmRfcHJlcHJvY2VzcwoKTE9HR0VSID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyZW5hX3Jhbmtlci5wcmVkaWN0IikKUFJPQkFCSUxJVFlfRVBTSUxPTiA9IDFlLTYKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICBDTEkKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBwYXJzZV9hcmdzKCkgLT4gYXJncGFyc2UuTmFtZXNwYWNlOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IkdlbmVyYXRlIHN1Ym1pc3Npb24gd2l0aCB0cmFpbmVkIFFMb1JBIEFyZW5hIHJhbmtlci4iCiAgICApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQtZGlyIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9Iuiuree7g+S6p+eJqeebruW9lSAo5YyF5ZCrIG1vZGVsLywgdG9rZW5pemVyLywgY29uZmlnLnlhbWwpIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1kaXIiLCB0eXBlPXN0ciwgZGVmYXVsdD0iLiIsCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9InRlc3QuY3N2IOaJgOWcqOebruW9lSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1wYXRoIiwgdHlwZT1zdHIsIGRlZmF1bHQ9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0i6L6T5Ye6IHN1Ym1pc3Npb24uY3N2IOi3r+W+hCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoLXNpemUiLCB0eXBlPWludCwgZGVmYXVsdD00LAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSLmjqjnkIYgYmF0Y2ggc2l6ZSIpCiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgIOW3peWFt+WHveaVsAojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQoKZGVmIHNldHVwX2xvZ2dpbmcoKSAtPiBOb25lOgogICAgbG9nZ2luZy5iYXNpY0NvbmZpZygKICAgICAgICBsZXZlbD1sb2dnaW5nLklORk8sCiAgICAgICAgZm9ybWF0PSIlKGFzY3RpbWUpcyB8ICUobGV2ZWxuYW1lKXMgfCAlKG1lc3NhZ2UpcyIsCiAgICAgICAgZGF0ZWZtdD0iJUg6JU06JVMiLAogICAgKQoKCmRlZiBmb3JtYXRfc2Vjb25kcyhzZWNvbmRzOiBmbG9hdCkgLT4gc3RyOgogICAgdG90YWwgPSBtYXgoaW50KHNlY29uZHMpLCAwKQogICAgbSwgcyA9IGRpdm1vZCh0b3RhbCwgNjApCiAgICBoLCBtID0gZGl2bW9kKG0sIDYwKQogICAgaWYgaCA+IDA6CiAgICAgICAgcmV0dXJuIGYie2h9aCB7bX1tIHtzfXMiCiAgICBpZiBtID4gMDoKICAgICAgICByZXR1cm4gZiJ7bX1tIHtzfXMiCiAgICByZXR1cm4gZiJ7c31zIgoKCmRlZiBkZXNjcmliZV9kZXZpY2UoZGV2aWNlOiB0b3JjaC5kZXZpY2UpIC0+IHN0cjoKICAgIGlmIGRldmljZS50eXBlICE9ICJjdWRhIjoKICAgICAgICByZXR1cm4gIkNQVSIKICAgIG5hbWUgPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShkZXZpY2UpCiAgICBtZW0gPSB0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhkZXZpY2UpLnRvdGFsX21lbW9yeSAvIDEwMjQqKjMKICAgIHJldHVybiBmIntuYW1lfSAoe21lbTouMWZ9IEdCKSIKCgpkZWYgbm9ybWFsaXplX3Byb2JhYmlsaXRpZXMoCiAgICBsb2dpdHM6IHRvcmNoLlRlbnNvciwKICAgIGVwc2lsb246IGZsb2F0ID0gUFJPQkFCSUxJVFlfRVBTSUxPTiwKKSAtPiBucC5uZGFycmF5OgogICAgIiIibG9naXRzIOKGkiDoo4HliarlkI7nmoTmpoLnjofvvIjnoa7kv50gbG9nX2xvc3Mg5LiN5Lya54iG54K477yJ44CCIiIiCiAgICBwcm9icyA9IHRvcmNoLnNvZnRtYXgobG9naXRzLCBkaW09LTEpLmNwdSgpLm51bXB5KCkKICAgIHByb2JzID0gbnAuY2xpcChwcm9icywgZXBzaWxvbiwgMS4wIC0gZXBzaWxvbikKICAgIHByb2JzID0gcHJvYnMgLyBwcm9icy5zdW0oYXhpcz0tMSwga2VlcGRpbXM9VHJ1ZSkKICAgIHJldHVybiBwcm9icwoKCmRlZiBydW5faW5mZXJlbmNlKGxvYWRlcjogRGF0YUxvYWRlciwgbW9kZWwsIGRldmljZTogdG9yY2guZGV2aWNlKSAtPiBucC5uZGFycmF5OgogICAgIiIi5a+55LiA5LiqIERhdGFMb2FkZXIg5omn6KGM5a6M5pW05o6o55CG5bm26L+U5Zue5qaC546H55+p6Zi144CCIiIiCiAgICBhbGxfcHJvYnMgPSBbXQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGJhdGNoIGluIHRxZG0obG9hZGVyLCBkZXNjPSJwcmVkaWN0Iik6CiAgICAgICAgICAgIGJhdGNoID0ge2s6IHYudG8oZGV2aWNlKSBmb3IgaywgdiBpbiBiYXRjaC5pdGVtcygpfQogICAgICAgICAgICBvdXRwdXRzID0gbW9kZWwoKipiYXRjaCkKICAgICAgICAgICAgcHJvYnMgPSBub3JtYWxpemVfcHJvYmFiaWxpdGllcyhvdXRwdXRzLmxvZ2l0cykKICAgICAgICAgICAgYWxsX3Byb2JzLmFwcGVuZChwcm9icykKICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShhbGxfcHJvYnMsIGF4aXM9MCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojICDliqDovb3mjqjnkIbmqKHlnosKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCmRlZiBsb2FkX2luZmVyZW5jZV9tb2RlbChjaGVja3BvaW50X2RpcjogUGF0aCwgY29uZmlnLCB0b2tlbml6ZXIpOgogICAgIiIiCiAgICDliqDovb3orq3nu4Plpb3nmoQgUUxvUkEg5qih5Z6L55So5LqO5o6o55CG44CCCgogICAg5q2l6aqk77yaCiAgICAgIDEuIOmHjeaWsOWKoOi9veWfuuW6p+aooeWeiyAoNC1iaXQg6YeP5YyWKQogICAgICAyLiDku44gYWRhcHRlciDnm67lvZXliqDovb0gTG9SQSDmnYPph40gKyBzY29yZSDliIbnsbvlpLQKICAgICAgMy4g5a+56b2QIHBhZF90b2tlbl9pZAogICAgIiIiCiAgICBtYyA9IGNvbmZpZy5tb2RlbAogICAgYWRhcHRlcl9kaXIgPSBjaGVja3BvaW50X2RpciAvICJtb2RlbCIKCiAgICAjIOmHj+WMlumFjee9ru+8iDQtYml0IOmcgOimgSBDVURB77yJCiAgICBibmJfY29uZmlnID0gTm9uZQogICAgaWYgbWMubG9hZF9pbl80Yml0IGFuZCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIGJuYl9jb25maWcgPSBCaXRzQW5kQnl0ZXNDb25maWcoCiAgICAgICAgICAgIGxvYWRfaW5fNGJpdD1UcnVlLAogICAgICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPW1jLmJuYl80Yml0X3F1YW50X3R5cGUsCiAgICAgICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNiwKICAgICAgICAgICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudD1tYy5ibmJfNGJpdF91c2VfZG91YmxlX3F1YW50LAogICAgICAgICkKICAgIGVsaWYgbWMubG9hZF9pbl80Yml0OgogICAgICAgIExPR0dFUi53YXJuaW5nKCLmnKrmo4DmtYvliLAgQ1VEQe+8jOi3s+i/hyA0LWJpdCDph4/ljJYiKQoKICAgICMg5Yqg6L296YWN572u5bm25aSE55CGIFZMTSBudW1fbGFiZWxzIOS8oOaSrQogICAgbW9kZWxfY29uZmlnID0gQXV0b0NvbmZpZy5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgbWMubW9kZWxfbmFtZSwKICAgICAgICBudW1fbGFiZWxzPU5VTV9MQUJFTFMsCiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICBjYWNoZV9kaXI9bWMuY2FjaGVfZGlyLAogICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9bWMubG9jYWxfZmlsZXNfb25seSwKICAgICkKICAgIGlmIGhhc2F0dHIobW9kZWxfY29uZmlnLCAidGV4dF9jb25maWciKToKICAgICAgICBtb2RlbF9jb25maWcudGV4dF9jb25maWcubnVtX2xhYmVscyA9IE5VTV9MQUJFTFMKCiAgICAjIOWvuem9kCBwYWRfdG9rZW5faWTvvIjliIbnsbvmqKHlnovpnIDopoHvvIkKICAgIGlmIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWxfY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsX2NvbmZpZywgInRleHRfY29uZmlnIik6CiAgICAgICAgICAgIG1vZGVsX2NvbmZpZy50ZXh0X2NvbmZpZy5wYWRfdG9rZW5faWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCgogICAgIyDliqDovb3ln7rluqfmqKHlnosKICAgIGR0eXBlID0gdG9yY2guZmxvYXQxNiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgdG9yY2guZmxvYXQzMgogICAgYmFzZV9tb2RlbCA9IEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1jLm1vZGVsX25hbWUsCiAgICAgICAgY29uZmlnPW1vZGVsX2NvbmZpZywKICAgICAgICBxdWFudGl6YXRpb25fY29uZmlnPWJuYl9jb25maWcsCiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICBjYWNoZV9kaXI9bWMuY2FjaGVfZGlyLAogICAgICAgIGxvY2FsX2ZpbGVzX29ubHk9bWMubG9jYWxfZmlsZXNfb25seSwKICAgICAgICB0b3JjaF9kdHlwZT1kdHlwZSwKICAgICkKCiAgICAjIOWKoOi9vSBMb1JBIGFkYXB0ZXIKICAgIGlmIG1jLnVzZV9sb3JhIGFuZCBhZGFwdGVyX2Rpci5leGlzdHMoKToKICAgICAgICBtb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZV9tb2RlbCwgc3RyKGFkYXB0ZXJfZGlyKSkKICAgICAgICBMT0dHRVIuaW5mbygi5bey5Yqg6L29IExvUkEgYWRhcHRlcjogJXMiLCBhZGFwdGVyX2RpcikKICAgIGVsc2U6CiAgICAgICAgbW9kZWwgPSBiYXNlX21vZGVsCiAgICAgICAgTE9HR0VSLmluZm8oIuacquS9v+eUqCBMb1JBIGFkYXB0ZXLvvIzliqDovb3lrozmlbTmqKHlnosiKQoKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyAg5Li75Ye95pWwCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBzZXR1cF9sb2dnaW5nKCkKICAgIGFyZ3MgPSBwYXJzZV9hcmdzKCkKICAgIGNoZWNrcG9pbnRfZGlyID0gUGF0aChhcmdzLmNoZWNrcG9pbnRfZGlyKQogICAgc3RhcnRlZF9hdCA9IHRpbWUucGVyZl9jb3VudGVyKCkKCiAgICAjIC0tLS0gMS4g5Yqg6L296YWN572uIC0tLS0KICAgIGNvbmZpZ19wYXRoID0gY2hlY2twb2ludF9kaXIgLyAiY29uZmlnLnlhbWwiCiAgICBpZiBub3QgY29uZmlnX3BhdGguZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYi57y65bCR6YWN572u5paH5Lu2OiB7Y29uZmlnX3BhdGh9IikKICAgIGNvbmZpZyA9IGxvYWRfY29uZmlnKGNvbmZpZ19wYXRoKQoKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgTE9HR0VSLmluZm8oIumihOa1i+WQr+WKqCIpCiAgICBMT0dHRVIuaW5mbygiY2hlY2twb2ludDogJXMiLCBjaGVja3BvaW50X2RpcikKICAgIExPR0dFUi5pbmZvKCLorr7lpIc6ICVzIiwgZGVzY3JpYmVfZGV2aWNlKGRldmljZSkpCgogICAgIyAtLS0tIDIuIOWKoOi9vSB0b2tlbml6ZXIgLS0tLQogICAgdG9rZW5pemVyX2RpciA9IGNoZWNrcG9pbnRfZGlyIC8gInRva2VuaXplciIKICAgIGlmIG5vdCB0b2tlbml6ZXJfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIue8uuWwkSB0b2tlbml6ZXIg55uu5b2VOiB7dG9rZW5pemVyX2Rpcn0iKQogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgc3RyKHRva2VuaXplcl9kaXIpLCB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgKQogICAgaWYgdG9rZW5pemVyLnBhZF90b2tlbiBpcyBOb25lOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAibGVmdCIKCiAgICAjIC0tLS0gMy4g5Yqg6L295qih5Z6LIC0tLS0KICAgIG1vZGVsID0gbG9hZF9pbmZlcmVuY2VfbW9kZWwoY2hlY2twb2ludF9kaXIsIGNvbmZpZywgdG9rZW5pemVyKQoKICAgICMgLS0tLSA0LiDliqDovb3mtYvor5XmlbDmja4gLS0tLQogICAgdGVzdF9jc3YgPSBQYXRoKGFyZ3MuZGF0YV9kaXIpIC8gY29uZmlnLmRhdGEudGVzdF9wYXRoCiAgICBMT0dHRVIuaW5mbygi5Yqg6L295rWL6K+V5pWw5o2uOiAlcyIsIHRlc3RfY3N2KQogICAgdGVzdF9kZiA9IGxvYWRfYW5kX3ByZXByb2Nlc3MoCiAgICAgICAgc3RyKHRlc3RfY3N2KSwKICAgICAgICBtYXhfY2hhcnM9Y29uZmlnLmRhdGEudGV4dF9tYXhfY2hhcnMsCiAgICAgICAgaXNfdHJhaW49RmFsc2UsCiAgICApCiAgICB0ZXN0X2lkcyA9IHRlc3RfZGZbImlkIl0udG9saXN0KCkKCiAgICB0ZXN0X2RhdGFzZXQgPSBidWlsZF9kYXRhc2V0KAogICAgICAgIHRlc3RfZGYsIHRva2VuaXplciwgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsIGlzX3RyYWluPUZhbHNlLAogICAgKQogICAgTE9HR0VSLmluZm8oIuW+hemihOa1i+agt+acrDogJXMiLCBsZW4odGVzdF9kYXRhc2V0KSkKCiAgICAjIC0tLS0gNS4g5o6o55CGIC0tLS0KICAgIGRhdGFfY29sbGF0b3IgPSBEYXRhQ29sbGF0b3JXaXRoUGFkZGluZyh0b2tlbml6ZXI9dG9rZW5pemVyLCBwYWRkaW5nPVRydWUpCiAgICBsb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIHRlc3RfZGF0YXNldCwKICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICBzaHVmZmxlPUZhbHNlLAogICAgICAgIGNvbGxhdGVfZm49ZGF0YV9jb2xsYXRvciwKICAgICkKCiAgICBhbGxfcHJvYnMgPSBydW5faW5mZXJlbmNlKGxvYWRlciwgbW9kZWwsIGRldmljZSkKCiAgICBpZiBjb25maWcuZGF0YS5pbmNsdWRlX3N3YXBfdHRhOgogICAgICAgIHN3YXBfZGF0YXNldCA9IGJ1aWxkX2RhdGFzZXQoCiAgICAgICAgICAgIHRlc3RfZGYsCiAgICAgICAgICAgIHRva2VuaXplciwKICAgICAgICAgICAgY29uZmlnLm1vZGVsLm1heF9sZW5ndGgsCiAgICAgICAgICAgIGlzX3RyYWluPUZhbHNlLAogICAgICAgICAgICBzd2FwX3BhaXJzPVRydWUsCiAgICAgICAgKQogICAgICAgIHN3YXBfbG9hZGVyID0gRGF0YUxvYWRlcigKICAgICAgICAgICAgc3dhcF9kYXRhc2V0LAogICAgICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgc2h1ZmZsZT1GYWxzZSwKICAgICAgICAgICAgY29sbGF0ZV9mbj1kYXRhX2NvbGxhdG9yLAogICAgICAgICkKICAgICAgICBMT0dHRVIuaW5mbygi5ZCv55SoIHN3YXAgVFRB77yM5YaN6L+Q6KGM5LiA5qyh5a+556ew6aKE5rWLIikKICAgICAgICBzd2FwX3Byb2JzID0gcnVuX2luZmVyZW5jZShzd2FwX2xvYWRlciwgbW9kZWwsIGRldmljZSkKICAgICAgICBzd2FwX3Byb2JzID0gc3dhcF9wcm9ic1s6LCBbMSwgMCwgMl1dCiAgICAgICAgYWxsX3Byb2JzID0gKGFsbF9wcm9icyArIHN3YXBfcHJvYnMpIC8gMi4wCgogICAgIyAtLS0tIDYuIOWGmeWHuiBzdWJtaXNzaW9uLmNzdiAtLS0tCiAgICByb3dzID0gW10KICAgIGZvciBzYW1wbGVfaWQsIHByb2IgaW4gemlwKHRlc3RfaWRzLCBhbGxfcHJvYnMsIHN0cmljdD1UcnVlKToKICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAgICAgICAgICJpZCI6IHNhbXBsZV9pZCwKICAgICAgICAgICAgSURfVE9fTEFCRUxbMF06IHByb2JbMF0sCiAgICAgICAgICAgIElEX1RPX0xBQkVMWzFdOiBwcm9iWzFdLAogICAgICAgICAgICBJRF9UT19MQUJFTFsyXTogcHJvYlsyXSwKICAgICAgICB9KQoKICAgIG91dHB1dF9wYXRoID0gKAogICAgICAgIFBhdGgoYXJncy5vdXRwdXRfcGF0aCkKICAgICAgICBpZiBhcmdzLm91dHB1dF9wYXRoCiAgICAgICAgZWxzZSBjaGVja3BvaW50X2RpciAvICJzdWJtaXNzaW9uLmNzdiIKICAgICkKICAgIHBkLkRhdGFGcmFtZShyb3dzKS50b19jc3Yob3V0cHV0X3BhdGgsIGluZGV4PUZhbHNlKQogICAgTE9HR0VSLmluZm8oCiAgICAgICAgIumihOa1i+WujOaIkDogJXMg6KGMLCDogJfml7Y9JXMiLAogICAgICAgIGxlbihyb3dzKSwgZm9ybWF0X3NlY29uZHModGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWRfYXQpLAogICAgKQogICAgTE9HR0VSLmluZm8oInN1Ym1pc3Npb24g5bey5L+d5a2Y5YiwOiAlcyIsIG91dHB1dF9wYXRoKQogICAgcHJpbnQoZiJzYXZlZCBzdWJtaXNzaW9uIHRvIHtvdXRwdXRfcGF0aH0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
}

for _name, _b64 in _FILES.items():
    (PKG_DIR / _name).write_text(
        base64.b64decode(_b64).decode('utf-8'),
        encoding='utf-8',
    )
    print(f"  写入 {_name}")

import sys
if str(PKG_DIR.parent) not in sys.path:
    sys.path.insert(0, str(PKG_DIR.parent))

print("\narena_ranker 已写入:", PKG_DIR)
print("sys.path 已更新")

In [ ]:
import arena_ranker
print('导入成功:', arena_ranker.__file__)

## 3. 配置

设置竞赛数据路径和训练参数。

⚠️ **请根据你的实际竞赛修改 `COMPETITION_SLUG`。**

T4 和 P100 都是 16GB 显存，对 0.8B 模型 VRAM 参数相同；
P100 没有 Tensor Core，训练速度会慢一些，但不需要改参数。

In [ ]:
import torch

# ============================================================
# 🔧 根据你的情况修改以下参数
# ============================================================
COMPETITION_SLUG = "llm-classification-finetuning"   # ← 改成你的竞赛 slug
MODEL_NAME       = "Qwen/Qwen3-0.6B"              # 基座模型
EPOCHS           = 3
BATCH_SIZE       = 2        # T4/P100 16GB 推荐 2
GRAD_ACCUM_STEPS = 8
MAX_LENGTH       = 1024     # T4/P100 16GB 推荐 1024
LEARNING_RATE    = 2e-4
USE_LORA         = True
LOAD_IN_4BIT     = True     # QLoRA 4-bit 量化
# ============================================================

DATA_DIR    = f"/kaggle/input/datasets/dannyatkaggle/llm-finetune-test1/"
OUTPUT_DIR  = "/kaggle/working/artifacts/default"

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"设备: {device_name} ({vram_gb:.1f} GB)")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")


In [ ]:
import os

data_files = os.listdir(DATA_DIR)
print("竞赛数据文件:", data_files)
assert "train.csv" in data_files, f"找不到 train.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
assert "test.csv"  in data_files, f"找不到 test.csv，请检查 COMPETITION_SLUG。当前目录: {DATA_DIR}"
print("✅ 数据验证通过")


## 4. 训练

使用 QLoRA 微调 `Qwen3.5-0.8B`，通过 HuggingFace Trainer 训练。

| 参数 | T4 / P100 16GB | 8GB 显存 | 说明 |
| --- | --- | --- | --- |
| `BATCH_SIZE` | 2 | 1 | per device |
| `GRAD_ACCUM_STEPS` | 8 | 16 | 有效 batch = BATCH_SIZE × steps |
| `MAX_LENGTH` | 1024 | 512 | 输入序列最大 token 数 |
| `EPOCHS` | 3 | 3 | 训练轮数 |
| `LEARNING_RATE` | 2e-4 | 2e-4 | AdamW 学习率 |
| `LOAD_IN_4BIT` | True | True | 4-bit NF4 量化 (QLoRA) |

> **P100 注意事项**: P100 (Pascal) 没有 Tensor Core，FP16 矩阵运算比 T4 慢，
> 但 bitsandbytes 4-bit 量化和 FP16 混合精度均可正常使用。
> VRAM 参数与 T4 相同，主要差异体现在训练速度上。**不支持 bf16**（已是默认关闭）。

In [ ]:
import sys

sys.argv = [
    "arena-train",
    "--data-dir",        DATA_DIR,
    "--output-dir",      OUTPUT_DIR,
    "--model-name",      MODEL_NAME,
    "--epochs",          str(EPOCHS),
    "--batch-size",      str(BATCH_SIZE),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--max-length",      str(MAX_LENGTH),
    "--learning-rate",   str(LEARNING_RATE),
]

if not USE_LORA:
    sys.argv.append("--disable-lora")
if not LOAD_IN_4BIT:
    sys.argv.append("--no-4bit")

from arena_ranker.train import main as train_main
train_main()


## 5. 推理与生成提交文件

加载训练好的 QLoRA adapter + 分类头，对 `test.csv` 推理生成 `submission.csv`。

In [ ]:
import importlib, sys

SUBMISSION_PATH = "/kaggle/working/submission.csv"

sys.argv = [
    "arena-predict",
    "--checkpoint-dir", OUTPUT_DIR,
    "--data-dir",       DATA_DIR,
    "--output-path",    SUBMISSION_PATH,
    "--batch-size",     "4",
]

from arena_ranker.predict import main as predict_main
predict_main()


## 6. 检查提交文件

In [ ]:
import pandas as pd

sub = pd.read_csv(SUBMISSION_PATH)
print(f"行数: {len(sub)}")
print(f"列名: {list(sub.columns)}")
print()
print(sub.head(10))
print()
print("各列概率统计:")
print(sub.describe())
print()

row_sums = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(f"概率行和范围: [{row_sums.min():.6f}, {row_sums.max():.6f}]")
print("✅ submission.csv 已生成:", SUBMISSION_PATH)


## 附录 A：离线模式（无需联网）

如果 notebook 不能联网（例如最终提交时），需要提前将模型上传到 Kaggle。

### 步骤

1. **上传模型到 Kaggle**
   - 在本地下载好 `Qwen/Qwen3.5-0.8B` 的完整文件
   - 前往 [kaggle.com/models](https://kaggle.com/models) → New Model
   - 上传模型文件夹（包含 config.json, model.safetensors 等）
   - 或者使用 Kaggle Datasets 上传也可以

2. **在 notebook 中添加模型数据集**
   - 右侧 Add Input → 搜索你上传的模型

3. **修改配置**
   ```python
   MODEL_NAME = "/kaggle/input/<model-dataset-slug>"  # 改为本地路径
   ```

4. **训练时加上 `--local-files-only`**
   ```python
   sys.argv.append("--local-files-only")
   ```


## 附录 B：将代码上传为 Kaggle Dataset

如果不想在 notebook 里内联代码，可以把仓库上传为 Kaggle Dataset：

1. 打包源码：
   ```bash
   zip -r arena-ranker-code.zip src/ pyproject.toml README.md
   ```

2. 上传到 Kaggle Datasets

3. 在 notebook 中安装：
   ```python
   !pip install /kaggle/input/arena-ranker-code/
   ```

4. 使用 CLI 命令：
   ```python
   !arena-train --data-dir /kaggle/input/<slug>/ --output-dir /kaggle/working/artifacts/default
   !arena-predict --checkpoint-dir /kaggle/working/artifacts/default --data-dir /kaggle/input/<slug>/ --output-path /kaggle/working/submission.csv
   ```


## 附录 C：显存不足时的参数调整

如果遇到 OOM，优先按以下顺序调整：

1. 降低 `MAX_LENGTH`（如 512）
2. 降低 `BATCH_SIZE` 到 1
3. 提高 `GRAD_ACCUM_STEPS`
4. 确保 `LOAD_IN_4BIT = True`

```python
# 8GB 显存推荐参数
BATCH_SIZE   = 1
GRAD_ACCUM   = 16
MAX_LENGTH   = 512
LOAD_IN_4BIT = True
```